# Pressing metrics: Barça vs Atlético (first half)

Standalone stats notebook. It reads the folder written by **section 16 of the build-up pressure notebook** (`pressure_stats_input/`), so no video and no re-detection are needed.

**Run order:** run the setup cell once, then any metric cell on its own (each metric cell only needs the setup cell).

| # | Metric | Main output |
|---|---|---|
| 1 | Average defensive line depth per team | m from own goal |
| 2 | Pressing intensity | % of out-of-possession time spent pressing |
| 3 | Pressure by pitch zone (middle and attacking third) | counts and % |
| 4 | Pressure heatmap | where each team's pressure is concentrated |
| 5 | Pressing intensity over time | 5-minute bins, low-sample bins masked |
| 6 | PPDA | passes allowed per defensive action, own third excluded |
| 7 | Time-to-press after losing the ball | mean / median / min / max within 10 s |
| 8 | Pass success rate, under pressure vs not | by team |
| 9 | Defensive line height | last-1 and last-3 defenders, IQR outlier removal |
| 10 | Team compactness out of possession | depth, width, area |
| 11 | Compactness while pressing vs not | depth, width, area |
| 12 | Third x channel grid | where each team's pressure happens (3 x 3) |
| 13 | Zone intensity per time in zone | % of the time the ball is in a zone that the team is pressing |
| 14 | Press-to-regain rate | % of pressing episodes that end with the team getting the ball back within 5 s |
| 15 | Pressing time per player | seconds and share of the team's pressing |
| 16 | Where each player presses | third x channel per player |

(8b is a check on section 8.) Sections 15 and 16 need `defenders.parquet` from the export; the rest need only frames, positions, spells and roster.

**Orientation of every figure:** Barça attacks left → right and Atlético right → left (heatmaps, grids, minimaps). The team is found from the team names (set `LTR_TEAM_ID` in the setup cell to override); if the tracking has Barça attacking towards -x, the figures are rotated 180° so the rule still holds.

**Real-frame checks:** after each metric a cell renders about 3 real frames of the match (separate full frames, not a collage), annotated with that metric's numbers. **Photo frames only: no video and no GIF is written.** They need the match video and the original tracking tables: set the paths in `VERIFY` in *Setup 2*. Without them the checks print a message and are skipped; the metrics are not affected. The PNG files go to `<export folder>/../verification/<metric>/`.

**Definitions inherited from the build-up notebook**
- *Pressing / under pressure* (`frames.under_pressure`): at least one defender within 5 m of the ball carrier who is either within 2 m or closing at 1.5 m/s or more, held for 0.4 s in a row. It is a per-frame flag for the **defending** team (`def_team`).
- *Out of possession* = frames of the export in which the team is `def_team`. Stoppages are already excluded by the possession spells.
- Zones are thirds of the pitch in the **pressing team's own attacking direction** (own third 0-35 m, middle 35-70 m, attacking 70-105 m, measured from the pressing team's own goal).
- **Pressure in the pressing team's own third is not counted as pressing** (it is deep defending, not a press). This is one switch in the setup cell, `EXCLUDE_OWN_PRESS_UP_TO_M`, applied once to `frames.under_pressure`, so every metric below uses the same definition. The original flag is kept as `under_pressure_raw`. Set the switch to 16.5 to exclude only the own penalty box, or to `None` to count everything.

## Setup: load the export, parameters, helpers

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from IPython.display import display

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

# ------------------------------------------------------------------ where the export lives (written by section 16 of the build-up notebook)
EXPORT_DIR_OVERRIDE = None        # put a path here to skip the search below
_CANDIDATES = [
    Path("C:/Users/user/Desktop/Football_cv_project/football-pipeline (1)/project/new/press/buildup_output/pressure_stats_input"),
    Path.cwd() / "buildup_output" / "pressure_stats_input",
    Path.cwd() / "pressure_stats_input",
]
EXPORT_DIR = Path(EXPORT_DIR_OVERRIDE) if EXPORT_DIR_OVERRIDE else next((p for p in _CANDIDATES if (p / "meta.json").exists()), None)
assert EXPORT_DIR is not None and (EXPORT_DIR / "meta.json").exists(), (
    "pressure_stats_input/ not found. Run section 16 of the build-up notebook first, or set EXPORT_DIR_OVERRIDE above.")

# ------------------------------------------------------------------ parameters of THIS notebook
PRM = dict(
    # metric 5: time bins
    BIN_MINUTES=5, MIN_BIN_FRAMES=500,        # a team/bin with fewer out-of-possession frames than this is masked (500 frames = 20 s)
    # metric 9 and 10: outlier removal / shape
    IQR_MULT=1.5,
    BACK_N=3, FRONT_N=3, MIN_PLAYERS_FOR_SHAPE=6,
    # metric 7: time-to-press
    TTP_WINDOW_SEC=10.0,                      # search window after the turnover
    TTP_GRACE_SEC=0.0,                        # set to 1.0 to ignore pressure in the first second (the player who just lost it is still next to the new carrier)
    # metrics 6 and 8: pass detection from carrier transitions (same rules as the first pressure notebook)
    MIN_PASS_DISTANCE_M=3.0,                  # shorter carrier-to-carrier distance = duel / tackle, not a pass
    MAX_RECEPTION_GAP_SEC=2.0,                # longer gap = ball out of play, skipped
    MIN_HOLD_FRAMES_PASS=2,                   # ignores single-frame carrier flicker
    PRESSURE_LOOKBACK_SEC=1.0,                # a pass is "under pressure" if the opponent was pressing at any frame in this window before release
    # metric 4: heatmap
    HEATMAP_GRIDSIZE=25, BOUNDS_TOL_M=2.0,
    # metrics 12-16
    MIN_CELL_FRAMES=100,                      # an intensity cell with fewer ball-in-cell frames than this is greyed out (100 frames = 4 s)
    EPISODE_GAP_FRAMES=5, MIN_EPISODE_SEC=0.5,   # pressing episode: runs of pressing frames, gaps up to 5 frames bridged, at least 0.5 s
    REGAIN_WINDOW_SEC=5.0,                    # regain = pressing team has the ball within this long after the episode ends
    MIN_PLAYER_PRESS_FRAMES=50,               # players with fewer pressing frames are listed but not read into (50 frames = 2 s)
)
LEFT_IS_LOW_Y_ATT = True                      # from the build-up notebook's roster cell: in a team's attacking frame, low y = its LEFT side (flip if mirrored)

# ------------------------------------------------------------------ load
for _n in ["frames", "positions", "spells", "roster"]:
    assert (EXPORT_DIR / f"{_n}.parquet").exists(), f"{_n}.parquet is missing in {EXPORT_DIR}"
meta = json.loads((EXPORT_DIR / "meta.json").read_text())
frames = pd.read_parquet(EXPORT_DIR / "frames.parquet")
positions = pd.read_parquet(EXPORT_DIR / "positions.parquet")
spells = pd.read_parquet(EXPORT_DIR / "spells.parquet")
roster = pd.read_parquet(EXPORT_DIR / "roster.parquet")

_need = dict(frames=["frame_idx", "spell_id", "poss_team", "def_team", "under_pressure", "n_pressing", "carrier_track_id", "x", "y"],
             positions=["frame_idx", "track_id", "team", "role", "x", "y"],
             spells=["spell_id", "team", "start_frame", "end_frame", "start_type", "duration_sec"],
             roster=["track_id", "team", "role"])
for _n, _df in [("frames", frames), ("positions", positions), ("spells", spells), ("roster", roster)]:
    _miss = [c for c in _need[_n] if c not in _df.columns]
    assert not _miss, f"{_n}: columns missing {_miss} (found {list(_df.columns)})"

FPS = int(meta["fps"])
PITCH_LENGTH, PITCH_WIDTH = float(meta["pitch_length"]), float(meta["pitch_width"])
TEAM_NAMES = {int(k): str(v) for k, v in meta["team_names"].items()}
ATT_POS = {int(k): bool(v) for k, v in meta["team_attacks_positive_x"].items()}     # team attacks towards +x
THIRD_1_M = float(meta["params"].get("THIRD_1_M", PITCH_LENGTH / 3))
THIRD_2_M = float(meta["params"].get("THIRD_2_M", 2 * PITCH_LENGTH / 3))
TEAMS = sorted(TEAM_NAMES)
ZONES = ["own_third", "middle_third", "attacking_third"]
# pressure with the ball closer than this to the PRESSING team's own goal is not counted as pressing:
#   THIRD_1_M (default) = the whole own third | 16.5 = only the own penalty box (what the first pressure notebook did) | None = count everything
EXCLUDE_OWN_PRESS_UP_TO_M = THIRD_1_M
PRESS_ZONES = [z for z in ZONES if not (z == "own_third" and EXCLUDE_OWN_PRESS_UP_TO_M is not None and EXCLUDE_OWN_PRESS_UP_TO_M >= THIRD_1_M)]   # thirds where pressing is counted
F = lambda sec: int(round(sec * FPS))
name = lambda t: TEAM_NAMES.get(int(t), str(t))

# ------------------------------------------------------------------ orientation of EVERY figure: Barca attacks LEFT -> RIGHT, the other team RIGHT -> LEFT
import re
LTR_TEAM_ID = None            # team id drawn attacking left -> right (Barcelona). None = detect it from the team names
_barca = [t for t in TEAMS if re.search(r"bar|fcb", TEAM_NAMES[t].lower())]
LTR_TEAM = LTR_TEAM_ID if LTR_TEAM_ID is not None else (_barca[0] if len(_barca) == 1 else None)
if LTR_TEAM is None:
    LTR_TEAM = next((t for t in TEAMS if ATT_POS[t]), TEAMS[0])
    print(f"WARNING: could not tell which team is Barca from {TEAM_NAMES}; drawing {name(LTR_TEAM)} left to right. Set LTR_TEAM_ID above.")
RTL_TEAM = next(t for t in TEAMS if t != LTR_TEAM)
FIG_TEAMS = [LTR_TEAM, RTL_TEAM]
DISPLAY_ROTATE = not ATT_POS[LTR_TEAM]       # tracking has Barca attacking towards -x: every figure is rotated 180 degrees so Barca still goes left -> right
TEAM_COLOR = {LTR_TEAM: "#1D9E75", RTL_TEAM: "#534AB7"}
TEAM_CMAP = {LTR_TEAM: "Greens", RTL_TEAM: "Purples"}

def to_display(x, y):
    """raw pitch coordinates -> figure coordinates (Barca attacks left -> right, the other team right -> left)."""
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    return (PITCH_LENGTH - x, PITCH_WIDTH - y) if DISPLAY_ROTATE else (x, y)

# ------------------------------------------------------------------ clean types
frames = frames.sort_values("frame_idx").drop_duplicates("frame_idx").reset_index(drop=True)
frames["poss_team"] = frames["poss_team"].astype(int)
frames["def_team"] = frames["def_team"].astype(int)
frames["under_pressure"] = frames["under_pressure"].astype(bool)
positions["team"] = positions["team"].astype(int)
positions["track_id"] = positions["track_id"].astype(int)
spells["team"] = spells["team"].astype(int)
roster["team"] = roster["team"].astype(int)

# ------------------------------------------------------------------ helpers
def team_frame(x, y, team):
    """Pitch metres -> the given team's own attacking frame (x = metres from that team's own goal, always attacking towards +x)."""
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    team = np.broadcast_to(np.asarray(team), x.shape)
    attacks_pos = pd.Series(team).map(ATT_POS).to_numpy(dtype=bool)
    return np.where(attacks_pos, x, PITCH_LENGTH - x), np.where(attacks_pos, y, PITCH_WIDTH - y)

def zone_of(x_own_frame):
    return pd.cut(pd.Series(np.asarray(x_own_frame, dtype=float)), [-np.inf, THIRD_1_M, THIRD_2_M, np.inf],
                  labels=ZONES, right=False)

def wilson(k, n, z=1.96):
    if n == 0:
        return np.nan, np.nan
    p = k / n; den = 1 + z ** 2 / n
    c = (p + z ** 2 / (2 * n)) / den
    h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / den
    return c - h, c + h

def iqr_clean(df, col, by="team", mult=None):
    """Drop values outside [Q1 - mult*IQR, Q3 + mult*IQR], computed separately for each team."""
    mult = PRM["IQR_MULT"] if mult is None else mult
    parts = []
    for _, g in df.groupby(by):
        q1, q3 = g[col].quantile([0.25, 0.75]); iqr = q3 - q1
        parts.append(g[(g[col] >= q1 - mult * iqr) & (g[col] <= q3 + mult * iqr)])
    return pd.concat(parts)

def draw_pitch(ax, lw=1.2):
    ax.add_patch(patches.Rectangle((0, 0), PITCH_LENGTH, PITCH_WIDTH, fill=False, color="black", linewidth=lw))
    ax.axvline(PITCH_LENGTH / 2, color="black", linewidth=0.6)
    ax.add_patch(patches.Circle((PITCH_LENGTH / 2, PITCH_WIDTH / 2), 9.15, fill=False, color="black", linewidth=0.6))
    bd, bw = 16.5, 40.3
    ax.add_patch(patches.Rectangle((0, (PITCH_WIDTH - bw) / 2), bd, bw, fill=False, color="black", linewidth=0.6))
    ax.add_patch(patches.Rectangle((PITCH_LENGTH - bd, (PITCH_WIDTH - bw) / 2), bd, bw, fill=False, color="black", linewidth=0.6))
    for gx in (THIRD_1_M, THIRD_2_M):
        ax.axvline(gx, color="grey", linewidth=0.5, linestyle=":")
    ax.set_xlim(-3, PITCH_LENGTH + 3); ax.set_ylim(-3, PITCH_WIDTH + 3); ax.set_aspect("equal"); ax.axis("off")

# ------------------------------------------------------------------ own-third pressure is not pressing: applied ONCE here, every metric below uses it
frames["under_pressure_raw"] = frames["under_pressure"]
if EXCLUDE_OWN_PRESS_UP_TO_M is not None:
    _px = team_frame(frames["x"], frames["y"], frames["def_team"])[0]
    frames["under_pressure"] = frames["under_pressure_raw"] & ~(_px < EXCLUDE_OWN_PRESS_UP_TO_M)

_CACHE = {}
def cached(fn):
    """Each derived table is built once, on first use, so every metric cell can be run on its own."""
    def wrap():
        if fn.__name__ not in _CACHE:
            _CACHE[fn.__name__] = fn()
        return _CACHE[fn.__name__]
    wrap.__name__ = fn.__name__
    return wrap

@cached
def defending_positions():
    """Outfield players of the DEFENDING team in every frame (= the team out of possession), with distance from their own goal."""
    pl = positions[positions["role"] == "player"]
    d = pl.merge(frames[["frame_idx", "def_team", "under_pressure"]], on="frame_idx", how="inner")
    d = d[d["team"] == d["def_team"]].copy()
    d["dist_own_goal"] = np.clip(team_frame(d["x"], d["y"], d["team"])[0], 0, PITCH_LENGTH)
    return d

@cached
def line_frames():
    """Per frame and out-of-possession team: height of the last defender (last1) and mean of the last three (last3), m from own goal."""
    d = defending_positions().sort_values(["frame_idx", "team", "dist_own_goal"])
    keys = ["frame_idx", "team"]
    rank = d.groupby(keys).cumcount()
    last1 = d.loc[rank == 0].set_index(keys)["dist_own_goal"].rename("last1")
    g3 = d.loc[rank < 3].groupby(keys)["dist_own_goal"].agg(["mean", "size"])
    last3 = g3.loc[g3["size"] == 3, "mean"].rename("last3")
    out = pd.concat([last1, last3], axis=1).reset_index()
    return out.merge(frames[["frame_idx", "under_pressure"]], on="frame_idx", how="left")

@cached
def team_shape():
    """Per frame and out-of-possession team: back line, front line, depth, IQR-trimmed width and area (depth x width)."""
    keys = ["frame_idx", "team"]
    d = defending_positions().sort_values(keys + ["dist_own_goal"]).copy()
    gb = d.groupby(keys)
    d["rank"] = gb.cumcount()
    d["n"] = gb["dist_own_goal"].transform("size")
    d = d[d["n"] >= PRM["MIN_PLAYERS_FOR_SHAPE"]]
    back = d[d["rank"] < PRM["BACK_N"]].groupby(keys)["dist_own_goal"].mean().rename("back_line_m")
    front = d[d["rank"] >= d["n"] - PRM["FRONT_N"]].groupby(keys)["dist_own_goal"].mean().rename("front_line_m")
    q = d.groupby(keys)["y"].quantile([0.25, 0.75]).unstack()
    q.columns = ["q1", "q3"]
    d = d.join(q, on=keys)
    iqr = d["q3"] - d["q1"]; m = PRM["IQR_MULT"]
    kept = d[(d["y"] >= d["q1"] - m * iqr) & (d["y"] <= d["q3"] + m * iqr)]
    w = kept.groupby(keys)["y"].agg(["max", "min"])
    width = (w["max"] - w["min"]).rename("width_m")
    shape = pd.concat([back, front, width], axis=1).dropna().reset_index()
    shape["depth_m"] = shape["front_line_m"] - shape["back_line_m"]
    shape["area_m2"] = shape["depth_m"] * shape["width_m"]
    return shape.merge(frames[["frame_idx", "under_pressure"]], on="frame_idx", how="left")

@cached
def detect_passes():
    """Passes and defensive actions from carrier transitions (same rules as the first pressure notebook, built on the export tables).
    Returns dict(passes, actions, info). passes: completed / intercepted attempts. actions: interceptions and tackles by the defending team."""
    team_of = roster.drop_duplicates("track_id").set_index("track_id")["team"].to_dict()
    car = frames.dropna(subset=["carrier_track_id"])[["frame_idx", "carrier_track_id"]].copy()
    car["carrier_track_id"] = car["carrier_track_id"].astype(int)
    car["team"] = car["carrier_track_id"].map(team_of)
    n_no_team = int(car["team"].isna().sum())
    car = car.dropna(subset=["team"]); car["team"] = car["team"].astype(int)
    max_gap = F(PRM["MAX_RECEPTION_GAP_SEC"])
    new_run = (car["carrier_track_id"] != car["carrier_track_id"].shift()) | (car["frame_idx"].diff() > max_gap)
    car["run_id"] = new_run.cumsum()
    runs = (car.groupby("run_id").agg(track_id=("carrier_track_id", "first"), team=("team", "first"),
                                      start_frame=("frame_idx", "first"), end_frame=("frame_idx", "last"), n_frames=("frame_idx", "count"))
            .reset_index(drop=True))
    runs = runs[runs["n_frames"] >= PRM["MIN_HOLD_FRAMES_PASS"]].reset_index(drop=True)

    a, b = runs.iloc[:-1].reset_index(drop=True), runs.iloc[1:].reset_index(drop=True)
    pr_ = pd.DataFrame(dict(passer_track_id=a["track_id"], passer_team=a["team"], release_frame=a["end_frame"],
                            receiver_track_id=b["track_id"], receiver_team=b["team"], reception_frame=b["start_frame"]))
    pr_["gap_frames"] = pr_["reception_frame"] - pr_["release_frame"]
    n_pairs_all = len(pr_)
    pr_ = pr_[pr_["gap_frames"] <= max_gap]
    n_pairs_gap_ok = len(pr_)

    pos_k = positions[["frame_idx", "track_id", "x", "y", "role"]].drop_duplicates(["frame_idx", "track_id"])
    pr_ = pr_.merge(pos_k[["frame_idx", "track_id", "x", "y"]].rename(columns=dict(frame_idx="release_frame", track_id="passer_track_id", x="release_x", y="release_y")),
                    on=["release_frame", "passer_track_id"], how="left")
    pr_ = pr_.merge(pos_k.rename(columns=dict(frame_idx="reception_frame", track_id="receiver_track_id", x="receive_x", y="receive_y", role="receiver_role")),
                    on=["reception_frame", "receiver_track_id"], how="left")
    n_missing_pos = int(pr_[["release_x", "receive_x"]].isna().any(axis=1).sum())
    pr_ = pr_.dropna(subset=["release_x", "release_y", "receive_x", "receive_y"]).copy()

    same = (pr_["passer_team"] == pr_["receiver_team"]).to_numpy()
    n_gk = int((~same & (pr_["receiver_role"] == "goalkeeper").to_numpy()).sum())
    pr_ = pr_[~(~same & (pr_["receiver_role"] == "goalkeeper").to_numpy())].copy()      # keeper save / claim: not a pressing action
    same = (pr_["passer_team"] == pr_["receiver_team"]).to_numpy()
    pr_["dist_m"] = np.hypot(pr_["receive_x"] - pr_["release_x"], pr_["receive_y"] - pr_["release_y"])
    short = (pr_["dist_m"] < PRM["MIN_PASS_DISTANCE_M"]).to_numpy()

    tackles = pr_[~same & short]
    inter = pr_[~same & ~short]
    acts = pd.concat([tackles.assign(action_type="tackle"), inter.assign(action_type="interception")])
    actions = pd.DataFrame(dict(defending_team=acts["receiver_team"], attacking_team=acts["passer_team"], action_type=acts["action_type"],
                                frame=acts["release_frame"], x=acts["release_x"], y=acts["release_y"],
                                attacker_track_id=acts["passer_track_id"], defender_track_id=acts["receiver_track_id"],
                                reception_frame=acts["reception_frame"], x_end=acts["receive_x"], y_end=acts["receive_y"])).reset_index(drop=True)
    passes = pr_[~short].copy()
    passes["outcome"] = np.where(passes["passer_team"] == passes["receiver_team"], "completed", "intercepted")
    passes = passes.reset_index(drop=True)
    info = dict(carrier_runs=len(runs), transitions=n_pairs_all, within_gap=n_pairs_gap_ok, missing_positions=n_missing_pos,
                gk_claims_excluded=n_gk, carriers_without_team=n_no_team)
    return dict(passes=passes, actions=actions, info=info)

# ------------------------------------------------------------------ helpers for the zone / channel / player metrics (12-16)
CH_ORDER = ["y_low", "centre", "y_high"]
CH_LABEL = {"y_low": "left" if LEFT_IS_LOW_Y_ATT else "right", "centre": "centre", "y_high": "right" if LEFT_IS_LOW_Y_ATT else "left"}
ZONE_SHORT = {"own_third": "own", "middle_third": "mid", "attacking_third": "att"}

def channel_of(y_own_frame):
    return pd.cut(pd.Series(np.asarray(y_own_frame, dtype=float)), [-np.inf, PITCH_WIDTH / 3, 2 * PITCH_WIDTH / 3, np.inf],
                  labels=CH_ORDER, right=False)

ZONE_TICK = {"own_third": "own\nthird", "middle_third": "middle\nthird", "attacking_third": "attacking\nthird"}

def draw_grid(ax, M, texts, title, cmap="Greens", vmax=None, rtl=False):
    """Grid of the thirds that count as pressing (columns) x channels (rows). rtl=True draws it for a team attacking RIGHT -> LEFT
    (columns and rows reversed), so every grid matches the real pitch: Barca goes left -> right, the other team right -> left."""
    M = np.asarray(M, dtype=float); zones, chans = list(PRESS_ZONES), list(CH_ORDER)
    if rtl:
        M = M[::-1, ::-1]; texts = [row[::-1] for row in texts[::-1]]; zones = zones[::-1]; chans = chans[::-1]
    cm = plt.get_cmap(cmap).copy(); cm.set_bad("#e6e6e6")
    ax.imshow(np.ma.masked_invalid(M), origin="lower", cmap=cm, aspect="auto", vmin=0, vmax=vmax)
    top = vmax if vmax is not None else (np.nanmax(M) if np.isfinite(M).any() else 1.0)
    for i in range(len(chans)):
        for j in range(len(zones)):
            dark = np.isfinite(M[i, j]) and top > 0 and M[i, j] / top > 0.6          # white text on dark cells
            ax.text(j, i, texts[i][j], ha="center", va="center", fontsize=9, color="white" if dark else "black")
    ax.set_xticks(range(len(zones))); ax.set_xticklabels([ZONE_TICK[z] for z in zones], fontsize=8)
    ax.set_yticks(range(len(chans))); ax.set_yticklabels([CH_LABEL[c] for c in chans], fontsize=8)
    ax.set_title(title + ("   \u2190 attacks" if rtl else "   attacks \u2192"), fontsize=10)

@cached
def frames_zoned():
    """frames + where the pressed ball is, in the PRESSING (defending) team's attacking frame: press_x / press_y, third (zone) and channel."""
    fz = frames.copy()
    fz["press_x"], fz["press_y"] = team_frame(fz["x"], fz["y"], fz["def_team"])
    fz["zone"] = zone_of(fz["press_x"]).to_numpy()
    fz["channel"] = channel_of(fz["press_y"]).to_numpy()
    return fz

@cached
def player_press_frames():
    """One row per defending outfield player and frame, with his instantaneous press flag, the team-level pressing flag and where the ball is."""
    path = EXPORT_DIR / "defenders.parquet"
    assert path.exists(), "defenders.parquet is missing in the export folder (section 16 of the build-up notebook writes it)."
    d = pd.read_parquet(path)
    assert {"frame_idx", "track_id", "is_press"} <= set(d.columns), f"defenders.parquet needs frame_idx, track_id, is_press (found {list(d.columns)})"
    d = d[["frame_idx", "track_id", "is_press"]].copy()
    d["track_id"] = d["track_id"].astype(int); d["is_press"] = d["is_press"].fillna(False).astype(bool)
    d = d.merge(frames_zoned()[["frame_idx", "def_team", "under_pressure", "zone", "channel"]], on="frame_idx", how="inner")
    rs = roster.drop_duplicates("track_id").set_index("track_id")
    d["team"] = d["track_id"].map(rs["team"])
    d = d[d["team"] == d["def_team"]].copy()
    d["team"] = d["team"].astype(int)
    d["player"] = d["track_id"].map(rs["name"] if "name" in rs else pd.Series(dtype=object)).fillna("track " + d["track_id"].astype(str))
    d["number"] = d["track_id"].map(rs["number"]) if "number" in rs else np.nan
    d["pressing"] = d["is_press"] & d["under_pressure"]          # this defender is one of the pressers inside a sustained pressing phase
    return d

# ------------------------------------------------------------------ tables shared by a metric cell and its real-frame check
def pressed_at(team, release_frames):
    """True if the opponent of `team` was pressing `team` at any frame in [release - lookback, release]."""
    L = F(PRM["PRESSURE_LOOKBACK_SEC"]); n_f = int(frames["frame_idx"].max()) + 2
    flag = np.zeros(n_f + 1, dtype=np.int64)
    idx = frames.loc[frames["under_pressure"] & (frames["poss_team"] == team), "frame_idx"].to_numpy()
    flag[idx + 1] = 1
    cs = np.cumsum(flag)                                   # cs[k] = pressing frames with frame_idx < k
    rel = np.asarray(release_frames, dtype=int)
    return (cs[rel + 1] - cs[np.maximum(rel - L, 0)]) > 0

@cached
def pass_pressure_table():
    """Pass attempts (completed / intercepted) with `under_pressure` (opponent pressing in the lookback window) and `success`."""
    passes = detect_passes()["passes"].copy()
    passes["under_pressure"] = False
    for team in TEAMS:
        mk = (passes["passer_team"] == team).to_numpy()
        if mk.any():
            passes.loc[mk, "under_pressure"] = pressed_at(team, passes.loc[mk, "release_frame"])
    passes["success"] = passes["outcome"] == "completed"
    return passes

@cached
def turnover_table():
    """One row per turnover spell: the team that lost the ball, when the new spell started and when (if) the loser first pressed."""
    win, grace = F(PRM["TTP_WINDOW_SEC"]), F(PRM["TTP_GRACE_SEC"])
    tv = spells.loc[spells["start_type"] == "turnover", ["spell_id", "team", "start_frame", "duration_sec"]].copy()
    tv["losing_team"] = 1 - tv["team"]                      # the team that lost the ball = the team that has to press
    pf_ = frames.loc[frames["under_pressure"], ["frame_idx", "spell_id"]]
    m = tv.merge(pf_, on="spell_id", how="left")
    m = m[(m["frame_idx"] >= m["start_frame"] + grace) & (m["frame_idx"] <= m["start_frame"] + win)]
    tv["first_press_frame"] = tv["spell_id"].map(m.groupby("spell_id")["frame_idx"].min())
    tv["time_to_press_sec"] = (tv["first_press_frame"] - tv["start_frame"]) / FPS
    tv["pressed"] = tv["time_to_press_sec"].notna()
    tv["ended_before_window"] = (~tv["pressed"]) & (tv["duration_sec"] < PRM["TTP_WINDOW_SEC"])
    return tv

@cached
def ppda_table():
    """PPDA per team (own defensive third excluded for both the passes allowed and the defensive actions)."""
    res = detect_passes(); passes, actions = res["passes"], res["actions"]
    rows = []
    for team in TEAMS:
        opp = 1 - team
        p = passes[(passes["outcome"] == "completed") & (passes["passer_team"] == opp)]
        p_own = team_frame(p["release_x"], p["release_y"], team)[0] < THIRD_1_M          # in the defending team's own third
        a = actions[actions["defending_team"] == team]
        a_own = team_frame(a["x"], a["y"], team)[0] < THIRD_1_M
        n_pass, n_act = int((~p_own).sum()), int((~a_own).sum())
        rows.append(dict(team=name(team), passes_allowed=n_pass, defensive_actions=n_act,
                         interceptions=int(((a["action_type"] == "interception").to_numpy() & ~a_own).sum()),
                         tackles=int(((a["action_type"] == "tackle").to_numpy() & ~a_own).sum()),
                         PPDA=round(n_pass / n_act, 2) if n_act else np.nan,
                         excluded_own_third_passes=int(p_own.sum()), excluded_own_third_actions=int(a_own.sum())))
    return pd.DataFrame(rows).set_index("team")

@cached
def press_episodes():
    """Pressing episodes: runs of pressing frames inside one spell (short gaps bridged). `regain_frame` = first frame within the regain window
    in which the pressing team has the ball carrier (NaN = not regained); `regained_and_held` = it also starts a stable possession spell."""
    W, GAP, MIN_F = F(PRM["REGAIN_WINDOW_SEC"]), PRM["EPISODE_GAP_FRAMES"], F(PRM["MIN_EPISODE_SEC"])
    fz = frames_zoned().sort_values("frame_idx")
    team_of = roster.drop_duplicates("track_id").set_index("track_id")["team"].to_dict()
    car = frames.dropna(subset=["carrier_track_id"])[["frame_idx", "carrier_track_id"]].copy()
    car["team"] = car["carrier_track_id"].astype(int).map(team_of)
    car = car.dropna(subset=["team"])

    def true_runs(mask, fidx, spell):
        brk = np.r_[True, (np.diff(fidx) != 1) | (spell[1:] != spell[:-1]) | (mask[1:] != mask[:-1])]
        st = np.flatnonzero(brk); en = np.r_[st[1:] - 1, len(mask) - 1]
        return [(int(s_), int(e_)) for s_, e_ in zip(st, en) if mask[s_]]

    eps = []
    for T in TEAMS:
        s = fz[fz["def_team"] == T]
        f_, up, sp = s["frame_idx"].to_numpy(), s["under_pressure"].to_numpy(bool), s["spell_id"].to_numpy()
        npr, zone = s["n_pressing"].to_numpy(), s["zone"].to_numpy()
        merged = []
        for a, b in true_runs(up, f_, sp):
            if merged and sp[a] == sp[merged[-1][1]] and f_[a] - f_[merged[-1][1]] - 1 <= GAP:
                merged[-1][1] = b                                        # bridge a short gap inside the same spell
            else:
                merged.append([a, b])
        starts_T = np.sort(spells.loc[spells["team"] == T, "start_frame"].to_numpy())
        car_T = np.sort(car.loc[car["team"] == T, "frame_idx"].to_numpy(dtype=int))
        for a, b in merged:
            n = int(f_[b] - f_[a] + 1)
            if n < MIN_F:
                continue
            end = int(f_[b])
            j = np.searchsorted(car_T, end, side="right")                # first carrier frame of T after the episode ended
            rf = int(car_T[j]) if j < len(car_T) and car_T[j] <= end + W else np.nan
            eps.append(dict(team=T, spell_id=int(sp[a]), start_frame=int(f_[a]), end_frame=end, n_frames=n, max_pressers=int(npr[a:b + 1].max()),
                            start_zone=zone[a], regain_frame=rf, regained=bool(not np.isnan(rf)),
                            regained_and_held=bool(np.searchsorted(starts_T, end + W, side="right") - np.searchsorted(starts_T, end, side="right") > 0)))
    return pd.DataFrame(eps)

@cached
def player_press_table():
    """Per player: pressing time and share of the team's pressing (index: team id, player)."""
    d = player_press_frames()
    g = d.groupby(["team", "player"]).agg(number=("number", "first"), defending_frames=("pressing", "size"), pressing_frames=("pressing", "sum"))
    g["pressing_frames"] = g["pressing_frames"].astype(int)
    team_press_frames = frames[frames["under_pressure"]].groupby("def_team").size()
    team_presser_total = g.groupby(level="team")["pressing_frames"].transform("sum")
    team_idx = g.index.get_level_values("team")
    return pd.DataFrame({"number": g["number"], "defending_sec": (g["defending_frames"] / FPS).round(0), "pressing_sec": (g["pressing_frames"] / FPS).round(1),
                         "share_of_team_presser_time_pct": (100 * g["pressing_frames"] / team_presser_total).round(1),
                         "involved_in_pct_of_team_pressing": (100 * g["pressing_frames"] / team_press_frames.reindex(team_idx).to_numpy()).round(1),
                         "pressing_rate_pct": (100 * g["pressing_frames"] / g["defending_frames"]).round(1)})

@cached
def player_cell_table():
    """Per player: % of his pressing frames in each third x channel cell (index: team name, player)."""
    d = player_press_frames()
    pp = d[d["pressing"]].dropna(subset=["zone", "channel"])
    cells = [(z, ch) for z in PRESS_ZONES for ch in CH_ORDER]
    col_name = lambda z, ch: f"{ZONE_SHORT[z]}_{CH_LABEL[ch]}"
    minp = PRM["MIN_PLAYER_PRESS_FRAMES"]
    rows = []
    for (team, player), s in pp.groupby(["team", "player"]):
        n = len(s); row = dict(team=name(team), player=player, pressing_sec=round(n / FPS, 1), enough_data=n >= minp)
        cnt = s.groupby(["zone", "channel"]).size()
        for z, ch in cells:
            row[col_name(z, ch)] = round(100 * cnt.get((z, ch), 0) / n, 1)
        best = max(cells, key=lambda c: cnt.get(c, 0))
        row["main_zone"], row["main_channel"] = best
        row["main_cell"] = f"{best[0].replace('_third', '')} / {CH_LABEL[best[1]]}"
        row["pct_attacking_third"] = round(100 * (s["zone"] == "attacking_third").mean(), 1)
        rows.append(row)
    return pd.DataFrame(rows).sort_values(["team", "pressing_sec"], ascending=[True, False]).set_index(["team", "player"])


# ------------------------------------------------------------------ check
print("export folder:", EXPORT_DIR)
print(f"teams: {TEAM_NAMES} | attacks towards +x: {ATT_POS} | {FPS} fps | pitch {PITCH_LENGTH:.0f} x {PITCH_WIDTH:.0f} m")
print(f"frames (possession frames with a press target): {len(frames):,} = {len(frames) / FPS / 60:.1f} min | "
      f"match clock up to {frames['frame_idx'].max() / FPS / 60:.1f} min")
print(f"positions: {len(positions):,} rows | spells: {len(spells)} | roster: {len(roster)} tracks")
print("out-of-possession frames per team (def_team):", {name(t): int((frames['def_team'] == t).sum()) for t in TEAMS})
print("zone boundaries (m from the pressing team's own goal):", round(THIRD_1_M, 1), round(THIRD_2_M, 1))
print(f"figures: {name(LTR_TEAM)} attacks left -> right, {name(RTL_TEAM)} right -> left"
      + (" (tracking has Barca attacking towards -x, so figures are rotated 180 degrees)" if DISPLAY_ROTATE else ""))
_ex = (frames["under_pressure_raw"] & ~frames["under_pressure"]).groupby(frames["def_team"]).sum()
print(f"own-zone pressure not counted ({'switch off' if EXCLUDE_OWN_PRESS_UP_TO_M is None else f'ball within {EXCLUDE_OWN_PRESS_UP_TO_M:.1f} m of the pressing team own goal'}): "
      f"{ {name(t): int(_ex.get(t, 0)) for t in TEAMS} } frames | pressing counted in: {PRESS_ZONES}")

## Setup 2: real-frame verification (video)
Every metric below is followed by a **real-frame check**: about 3 separate, full frames of the match (not a collage), annotated with that metric's own numbers. No video and no GIF is produced, only photo frames (PNG). The annotations are drawn on the grass with the per-frame camera calibration, hidden behind the players, and a small minimap in the corner shows the same thing in the figure orientation (Barça left → right).

It needs, on top of the export folder: the match video and the original tracking tables (pixel boxes and ball pixel position). Set the paths in `VERIFY` in the next cell. If they are not found the checks print a message and are skipped; the metrics themselves do not depend on them. Frames are written to `<export folder>/../verification/<metric>/`.

In [ ]:
# ==================================================================================================================================
# SETUP 2: real-frame verification. Every metric below gets about 3 REAL frames of the match (separate full frames, not a collage), annotated
# with that metric's numbers (photo frames only, no video or GIF). Needs the match video and the original tracking tables (pixel boxes) on top of the
# export folder. The camera calibration, number tags and ball marker are the ones of the build-up notebook (sections 10-14).
# ==================================================================================================================================
import time, traceback, importlib.util
from PIL import Image as PILImage, ImageDraw, ImageFont
import matplotlib
from IPython.display import Image as IPImage, Markdown

HAVE_CV2 = importlib.util.find_spec("cv2") is not None
if HAVE_CV2:
    import cv2

_BASE = "C:/Users/user/Desktop/Football_cv_project/football-pipeline (1)/project/barca_atletico_first_half"
VERIFY = dict(
    enabled=True,
    video_path=_BASE + "/clip.mkv",
    player_table=_BASE + "/new_cache/player_frame_table_named.parquet",   # boxes, feet pixels, pitch coordinates (the table the build-up notebook reads)
    ball_table=_BASE + "/new_cache/ball_frame_table_named.parquet",       # ball pixel position
    frame_offset=0,                  # video frame = frame_idx + frame_offset
    out_dir=None,                    # None = <export folder>/../verification
    pre_sec=1.0, post_sec=1.5,       # an example frame is only chosen where the pressing / defending state is stable this long around it
    still_width=950,                 # width of the frames shown in the notebook (the PNG files are full resolution)
)
ONLY = None                          # e.g. ["m09_line_height"] to render only some metrics

def _check_verify():
    problems = []
    if not VERIFY["enabled"]:
        problems.append("VERIFY['enabled'] is False")
    if not HAVE_CV2:
        problems.append("OpenCV is missing: pip install opencv-python, then restart the kernel")
    for k in ("video_path", "player_table", "ball_table"):
        if not Path(VERIFY[k]).exists():
            problems.append(f"{k} not found: {VERIFY[k]}")
    return problems

VERIFY_PROBLEMS = _check_verify()
VERIFY_READY = not VERIFY_PROBLEMS
VERIFY_DIR = Path(VERIFY["out_dir"]) if VERIFY["out_dir"] else EXPORT_DIR.parent / "verification"

TEAM_RGB = {LTR_TEAM: (30, 120, 255), RTL_TEAM: (235, 55, 55)}      # annotation colours (Barca blue, the other team red)
PRESS_RGB, OK_RGB, BAD_RGB, GREY_RGB = (255, 70, 70), (60, 200, 90), (240, 60, 60), (200, 200, 200)
ZONE_BOUNDS = {"own_third": (0.0, THIRD_1_M), "middle_third": (THIRD_1_M, THIRD_2_M), "attacking_third": (THIRD_2_M, PITCH_LENGTH)}
CH_BOUNDS = {"y_low": (0.0, PITCH_WIDTH / 3), "centre": (PITCH_WIDTH / 3, 2 * PITCH_WIDTH / 3), "y_high": (2 * PITCH_WIDTH / 3, PITCH_WIDTH)}
MAX_FRAME = int(frames["frame_idx"].max())

# ------------------------------------------------------------------ small geometry helpers shared by the real-frame checks
def raw_x(team, dist_from_own_goal):
    """distance from a team's own goal -> raw pitch x."""
    return dist_from_own_goal if ATT_POS[team] else PITCH_LENGTH - dist_from_own_goal

def raw_y(team, y_att):
    """y in a team's attacking frame -> raw pitch y."""
    return y_att if ATT_POS[team] else PITCH_WIDTH - y_att

def cell_rect(team, zone, channel):
    """(x0, y0, x1, y1) in raw pitch metres of a third x channel cell, as defined in the pressing team's attacking frame."""
    xs = sorted([raw_x(team, ZONE_BOUNDS[zone][0]), raw_x(team, ZONE_BOUNDS[zone][1])])
    ys = sorted([raw_y(team, CH_BOUNDS[channel][0]), raw_y(team, CH_BOUNDS[channel][1])]) if channel else [0.0, PITCH_WIDTH]
    return (xs[0], ys[0], xs[1], ys[1])

def mmss_clock(frame):
    s = frame / FPS
    return f"{int(s // 60):02d}:{s % 60:04.1f}"

def spread(candidates, n=3, gap_sec=20):
    """First n candidate frames in order of preference that are at least gap_sec apart."""
    out = []
    for f in candidates:
        if f is not None and all(abs(int(f) - g) >= F(gap_sec) for g in out):
            out.append(int(f))
        if len(out) == n:
            break
    return out

def typical(df):
    """The middle row (by frame) of a candidate table: a typical case rather than the first or the last one."""
    d = df.sort_values("frame_idx")
    return d.iloc[len(d) // 2]

def make_example(key, caption, f0=None, f1=None, **data):
    key = int(key)
    f0 = key - F(VERIFY["pre_sec"]) if f0 is None else int(f0)
    f1 = key + F(VERIFY["post_sec"]) if f1 is None else int(f1)
    return dict(key=key, f0=max(f0, 0), f1=min(f1, MAX_FRAME), caption=caption, **data)

# ------------------------------------------------------------------ lookups on the export tables
@cached
def stint_info():
    """Stints = unbroken stretches of frames with the same defending team inside one possession spell (a clip stays inside one)."""
    z = frames[["frame_idx", "spell_id", "def_team"]].sort_values("frame_idx").copy()
    brk = (z["frame_idx"].diff() != 1) | (z["spell_id"] != z["spell_id"].shift()) | (z["def_team"] != z["def_team"].shift())
    z["stint"] = brk.cumsum()
    g = z.groupby("stint")["frame_idx"].agg(s0="min", s1="max")
    return z.join(g, on="stint").set_index("frame_idx")[["stint", "s0", "s1"]]

@cached
def fzs():
    """frames_zoned + stint bounds + 'stable' flags (pressing / not pressing for +-0.5 s) + whether a clip window fits inside the stint."""
    z = frames_zoned().copy()
    z = z.join(stint_info()[["s0", "s1"]], on="frame_idx")
    up = z["under_pressure"].astype(int)
    h = F(0.5)
    inside = (z["frame_idx"] - h >= z["s0"]) & (z["frame_idx"] + h <= z["s1"])
    z["stable_press"] = (up.rolling(2 * h + 1, center=True, min_periods=2 * h + 1).min() == 1) & inside
    z["stable_nopress"] = (up.rolling(2 * h + 1, center=True, min_periods=2 * h + 1).max() == 0) & inside
    z["ok_clip"] = (z["frame_idx"] - F(VERIFY["pre_sec"]) >= z["s0"]) & (z["frame_idx"] + F(VERIFY["post_sec"]) <= z["s1"])
    return z

def pick_frame(pressing=True, team=None, extra=None, need_clip=True, prefer_n=2):
    """A typical frame in the middle of a stable pressing (or stable not-pressing) stretch, optionally for one team / a condition on fzs()."""
    z = fzs(); m = z["stable_press"] if pressing else z["stable_nopress"]
    if need_clip:
        m = m & z["ok_clip"]
    if team is not None:
        m = m & (z["def_team"] == team)
    if extra is not None:
        m = m & extra(z)
    c = z[m]
    if c.empty:
        return None
    if pressing and prefer_n:
        c2 = c[c["n_pressing"] >= prefer_n]
        c = c2 if len(c2) else c
    return int(typical(c)["frame_idx"])

_POS = {}
def pos_at(f):
    if "sorted" not in _POS:
        _POS["sorted"] = positions.sort_values("frame_idx").reset_index(drop=True)
        _POS["f"] = _POS["sorted"]["frame_idx"].to_numpy()
    lo, hi = np.searchsorted(_POS["f"], f, side="left"), np.searchsorted(_POS["f"], f, side="right")
    return _POS["sorted"].iloc[lo:hi]

def pressers_at(f):
    """Track ids of the defenders that are pressing at frame f (their own press flag inside a sustained pressing phase)."""
    if "press" not in _POS:
        try:
            d = player_press_frames()
            p = d[d["pressing"]][["frame_idx", "track_id"]].sort_values("frame_idx")
            _POS["press"], _POS["press_f"] = p, p["frame_idx"].to_numpy()
        except AssertionError:
            _POS["press"] = None
    if _POS["press"] is None:
        return []
    lo, hi = np.searchsorted(_POS["press_f"], f, side="left"), np.searchsorted(_POS["press_f"], f, side="right")
    return [int(t) for t in _POS["press"]["track_id"].to_numpy()[lo:hi]]

_BALLPX = {}
def ball_px():
    if "t" not in _BALLPX:
        b = pd.read_parquet(VERIFY["ball_table"], columns=["frame_idx", "ball_px_x", "ball_px_y", "ball_source", "carrier_track_id"])
        _BALLPX["t"] = b.drop_duplicates("frame_idx").set_index("frame_idx")
    return _BALLPX["t"]

class Ctx:
    pass

def make_ctx(f, bp):
    """Everything an overlay builder may want to know about one frame."""
    c = Ctx(); c.f = int(f)
    if "fz" not in _POS:
        _POS["fz"] = frames_zoned().set_index("frame_idx", drop=False)
    FZ = _POS["fz"]
    c.fr = FZ.loc[f] if f in FZ.index else None                  # None = this frame is not in the export (no possession / no press target)
    c.pos = pos_at(f)
    c.pos_xy = {int(r.track_id): (float(r.x), float(r.y)) for r in c.pos.itertuples()}
    c.pressers = pressers_at(f) if c.fr is not None else []
    car = None
    if c.fr is not None and pd.notna(c.fr["carrier_track_id"]):
        car = int(c.fr["carrier_track_id"])
    elif f in bp.index and pd.notna(bp.at[f, "carrier_track_id"]):
        car = int(bp.at[f, "carrier_track_id"])
    c.carrier = car
    c.tgt = (float(c.fr["x"]), float(c.fr["y"])) if c.fr is not None else None
    c.under = bool(c.fr["under_pressure"]) if c.fr is not None else False
    return c

def press_ops(ctx, show_lines=True):
    """Ground graphics of the pressing signal: 5 m and 2 m rings around the pressed target, and a line to each pressing defender."""
    ops, hl = [], {}
    if ctx.tgt is None:
        return ops, hl
    col = PRESS_RGB if ctx.under else GREY_RGB
    ops += [("ring", ctx.tgt, 5.0, col, 0.10 if ctx.under else 0.04, 0.9, 4), ("ring", ctx.tgt, 2.0, col, 0.0, 0.7, 2)]
    if show_lines:
        for tid in ctx.pressers:
            if tid in ctx.pos_xy:
                p = ctx.pos_xy[tid]; dist = float(np.hypot(p[0] - ctx.tgt[0], p[1] - ctx.tgt[1]))
                ops.append(("line", p, ctx.tgt, PRESS_RGB, 0.95, 4)); hl[tid] = (PRESS_RGB, f"{dist:.1f} m")
    return ops, hl

def defenders_sorted(ctx):
    """(defending team, its outfield players at this frame sorted by distance from own goal) or None."""
    if ctx.fr is None:
        return None
    t = int(ctx.fr["def_team"])
    d = ctx.pos[(ctx.pos["team"] == t) & (ctx.pos["role"] == "player")].copy()
    if len(d) < 3:
        return None
    d["dist"] = np.clip(team_frame(d["x"], d["y"], np.full(len(d), t))[0], 0, PITCH_LENGTH)
    return t, d.sort_values("dist")

def player_frame_pick(team, player, zone=None, channel=None):
    """A typical frame in the middle of a stretch where THIS player is pressing (his own flag, +-0.5 s), optionally with the ball in a given third / channel."""
    d = player_press_frames()
    dp = d[(d["team"] == team) & (d["player"] == player)].sort_values("frame_idx")
    h = F(0.5)
    share = dp["pressing"].astype(int).rolling(2 * h + 1, center=True, min_periods=2 * h + 1).mean()      # tolerates a flicker of his own flag
    stab = (share >= 0.7) & dp["pressing"]
    contig = (dp["frame_idx"].shift(-h) - dp["frame_idx"].shift(h)) == 2 * h
    ok = dp.assign(stab=stab & contig).merge(fzs()[["frame_idx", "ok_clip"]], on="frame_idx")
    m = ok["stab"] & ok["ok_clip"]
    if zone is not None:
        m &= ok["zone"] == zone
    if channel is not None:
        m &= ok["channel"] == channel
    c = ok[m]
    return None if c.empty else int(typical(c)["frame_idx"])

def dedupe(examples, gap_sec=8):
    """Drops examples whose key frame is within gap_sec of an earlier one, so the frames show different moments (and says so)."""
    out = []
    for e in examples:
        if e is None:
            continue
        near = [o for o in out if abs(e["key"] - o["key"]) < F(gap_sec)]
        if near:
            print(f"  (dropped an example at {mmss_clock(e['key'])}: within {gap_sec} s of another one)")
        else:
            out.append(e)
    return out

def block_geometry(ctx):
    """The defending team's block at this frame, computed exactly as in metrics 10 and 11 (back 3 / front 3 lines, IQR-trimmed width)."""
    di = defenders_sorted(ctx)
    if di is None or len(di[1]) < PRM["MIN_PLAYERS_FOR_SHAPE"]:
        return None
    t, d = di
    back, front = d["dist"].iloc[:PRM["BACK_N"]].mean(), d["dist"].iloc[-PRM["FRONT_N"]:].mean()
    y = d["y"].to_numpy(); q1, q3 = np.percentile(y, [25, 75]); iqr = q3 - q1
    yk = y[(y >= q1 - PRM["IQR_MULT"] * iqr) & (y <= q3 + PRM["IQR_MULT"] * iqr)]
    return dict(team=t, back=back, front=front, ymin=float(yk.min()), ymax=float(yk.max()), depth=front - back, width=float(yk.max() - yk.min()),
                area=(front - back) * float(yk.max() - yk.min()), players=d)

def block_ops(g):
    xs = sorted([raw_x(g["team"], g["back"]), raw_x(g["team"], g["front"])]); col = TEAM_RGB[g["team"]]
    return [("rect", (xs[0], g["ymin"], xs[1], g["ymax"]), col, 0.12, 0.95),
            ("line", (raw_x(g["team"], g["back"]), 0), (raw_x(g["team"], g["back"]), PITCH_WIDTH), col, 0.9, 4),
            ("line", (raw_x(g["team"], g["front"]), 0), (raw_x(g["team"], g["front"]), PITCH_WIDTH), col, 0.9, 4)]

def b_press(ctx, ex):
    """Generic overlay of the pressing signal (metrics 2, 3, 4, 5, 12, 13): rings and lines to the pressers, an optional shaded zone / cell,
    the example's own text lines and a PRESSING / NOT PRESSING chip that changes live."""
    ops, hl = press_ops(ctx)
    hud = list(ex.get("hud", []))
    if ctx.fr is None:
        return dict(hud=["(this frame is not in the export: no possession or no press target)"] + hud)
    t = int(ctx.fr["def_team"])
    n_p = len(ctx.pressers)
    hud.insert(0, f"{name(t)} out of possession: " + ((f"carrier pressed by {n_p} defender(s)" if n_p else "carrier pressed") if ctx.under else "carrier not pressed"))
    if ex.get("shade") == "zone" and ctx.fr["zone"] in PRESS_ZONES:
        ops.insert(0, ("rect", cell_rect(t, ctx.fr["zone"], None), TEAM_RGB[t], 0.10, 0.9))
        hud.append(f"ball zone now: {str(ctx.fr['zone']).replace('_', ' ')} ({float(ctx.fr['press_x']):.0f} m from {name(t)}'s own goal)")
    elif ex.get("rect") is not None and ctx.tgt is not None:
        r = ex["rect"]; ops.insert(0, ("rect", r, TEAM_RGB[ex["team"]], 0.12, 0.95))
        hud.append("ball " + ("INSIDE" if r[0] <= ctx.tgt[0] <= r[2] and r[1] <= ctx.tgt[1] <= r[3] else "outside") + " the highlighted cell")
    return dict(ops=ops, hl=hl, hud=hud, chip=("PRESSING", PRESS_RGB) if ctx.under else ("NOT PRESSING", (90, 130, 90)))

# ------------------------------------------------------------------ camera calibration (from the build-up notebook, section 13)
CALIB_COLS = ["frame_idx", "track_id", "role", "team", "number", "x1", "y1", "x2", "y2", "foot_px_x", "foot_px_y", "pitch_x_raw", "pitch_y_raw", "pitch_valid", "source"]

def fit_homographies(df, thr_px=6.0, min_pts=6):
    d_ = df[df["pitch_valid"].fillna(False).astype(bool) & df["pitch_x_raw"].notna() & df["foot_px_x"].notna()]
    out = {}
    for f, g in d_.groupby("frame_idx"):
        if len(g) < min_pts:
            continue
        src = g[["pitch_x_raw", "pitch_y_raw"]].to_numpy(np.float64); dst = g[["foot_px_x", "foot_px_y"]].to_numpy(np.float64)
        H, _ = cv2.findHomography(src, dst, cv2.RANSAC, thr_px)
        if H is None:
            continue
        err = np.hypot(*(cv2.perspectiveTransform(src.reshape(-1, 1, 2), H).reshape(-1, 2) - dst).T)
        ok = err < thr_px
        if ok.sum() < min_pts - 1:
            continue
        out[int(f)] = dict(H=H / H[2, 2], err=float(np.median(err[ok])), n=int(ok.sum()))
    return out

def smooth_homographies(raw, frames_, max_gap=12, win=5):
    """Fill short gaps and lightly smooth the 9 matrix entries over time (centred, so no lag)."""
    A = np.full((len(frames_), 9), np.nan)
    for i, f in enumerate(frames_):
        if int(f) in raw:
            A[i] = raw[int(f)]["H"].ravel()
    A = pd.DataFrame(A).interpolate(limit=max_gap, limit_area="inside").to_numpy().copy()
    S = pd.DataFrame(A).rolling(win, center=True, min_periods=1).mean().to_numpy().copy()
    S[np.isnan(A).any(axis=1)] = np.nan
    return {int(f): (S[i].reshape(3, 3) if not np.isnan(S[i]).any() else None) for i, f in enumerate(frames_)}

def proj(H, xy):
    """pitch metres (N,2) -> pixels (N,2); NaN where the point is behind the camera."""
    xy = np.asarray(xy, float).reshape(-1, 2)
    p = np.c_[xy, np.ones(len(xy))] @ H.T
    w = p[:, 2]
    with np.errstate(divide="ignore", invalid="ignore"):
        out = p[:, :2] / w[:, None]
    out[w <= 1e-9] = np.nan
    return out

def clip_poly(pts, W, H, m=60):
    """Sutherland-Hodgman clip of a polygon to the image (plus a margin), so far-away corners cannot distort it."""
    x0, y0, x1, y1 = -m, -m, W + m, H + m
    def step(poly, inside, inter):
        out = []
        for i in range(len(poly)):
            a, b = poly[i - 1], poly[i]
            ia, ib = inside(a), inside(b)
            if ia and ib:
                out.append(b)
            elif ia and not ib:
                out.append(inter(a, b))
            elif (not ia) and ib:
                out.append(inter(a, b)); out.append(b)
        return out
    poly = [tuple(map(float, p)) for p in pts]
    for inside, inter in (
        (lambda p: p[0] >= x0, lambda a, b: (x0, a[1] + (b[1] - a[1]) * (x0 - a[0]) / (b[0] - a[0]))),
        (lambda p: p[0] <= x1, lambda a, b: (x1, a[1] + (b[1] - a[1]) * (x1 - a[0]) / (b[0] - a[0]))),
        (lambda p: p[1] >= y0, lambda a, b: (a[0] + (b[0] - a[0]) * (y0 - a[1]) / (b[1] - a[1]), y0)),
        (lambda p: p[1] <= y1, lambda a, b: (a[0] + (b[0] - a[0]) * (y1 - a[1]) / (b[1] - a[1]), y1))):
        if not poly:
            break
        poly = step(poly, inside, inter)
    return poly

def smooth_boxes(boxes, win=5):
    """Centred moving average of each player's box and feet position inside a clip, so tags stay put instead of jittering."""
    cols = ["x1", "y1", "x2", "y2", "foot_px_x", "foot_px_y"]
    b = boxes.sort_values(["track_id", "frame_idx"]).copy()
    seg = (b.groupby("track_id")["frame_idx"].diff() != 1).cumsum()
    for c in cols:
        b[c] = b.groupby(["track_id", seg])[c].transform(lambda s_: s_.rolling(win, center=True, min_periods=1).mean())
    return b

def read_calib(f0, f1):
    p = VERIFY["player_table"]
    try:
        c = pd.read_parquet(p, columns=CALIB_COLS, filters=[("frame_idx", ">=", int(f0)), ("frame_idx", "<=", int(f1))])
    except Exception:
        c = pd.read_parquet(p, columns=CALIB_COLS); c = c[(c["frame_idx"] >= f0) & (c["frame_idx"] <= f1)]
    return c

# ------------------------------------------------------------------ drawing
def _find_font(bold):
    cands = (["C:/Windows/Fonts/segoeuib.ttf", "C:/Windows/Fonts/arialbd.ttf"] if bold else ["C:/Windows/Fonts/segoeui.ttf", "C:/Windows/Fonts/arial.ttf"])
    cands.append(str(Path(matplotlib.get_data_path()) / "fonts" / "ttf" / ("DejaVuSans-Bold.ttf" if bold else "DejaVuSans.ttf")))
    return next((c for c in cands if Path(c).exists()), None)

_FONTS = {}
def _font(size, bold=False):
    key = (int(size), bold)
    if key not in _FONTS:
        p = _find_font(bold)
        _FONTS[key] = ImageFont.truetype(p, int(size)) if p else ImageFont.load_default()
    return _FONTS[key]

def draw_ball(img, ball_xy, low_conf=False):
    """Ball marker of the build-up notebook: white ring + gold ring + centre dot, dashed ring when the position is interpolated."""
    if ball_xy is None:
        return
    cx, cy = int(round(ball_xy[0])), int(round(ball_xy[1])); r = max(int(round(7 * img.shape[0] / 1080)), 5)
    if low_conf:
        for i in range(0, 16, 2):
            cv2.ellipse(img, (cx, cy), (r, r), 0, 360 * i / 16, 360 * (i + 0.6) / 16, (0, 150, 210), 1, cv2.LINE_AA)
    else:
        cv2.circle(img, (cx, cy), r, (255, 255, 255), 2, cv2.LINE_AA)
        cv2.circle(img, (cx, cy), r - 1, (0, 220, 255), 1, cv2.LINE_AA)
        cv2.circle(img, (cx, cy), 1, (0, 220, 255), -1, cv2.LINE_AA)

def ground_layer(pil, H, ops, boxes_f):
    """Draws the ops on the grass with the per-frame camera mapping, hidden behind the players (as in the build-up notebook)."""
    if H is None or not ops:
        return pil
    Ww, Hh = pil.size; s = Hh / 1080
    ov = PILImage.new("RGBA", pil.size, (0, 0, 0, 0)); dr = ImageDraw.Draw(ov)
    A = lambda a: int(255 * a)
    labels = []

    def seg_m(p0, p1, rgba, width):
        q = proj(H, [p0, p1])
        if np.isnan(q).any():
            return
        c = lambda v: tuple(int(t) for t in np.clip(v, -1e7, 1e7))
        ok, a, b = cv2.clipLine((0, 0, Ww, Hh), c(q[0]), c(q[1]))
        if ok:
            dr.line([a, b], fill=rgba, width=width)

    def poly_m(pts_m, fill, outline, width):
        pts_m = [tuple(map(float, p)) for p in pts_m]
        p = proj(H, pts_m)
        if np.isnan(p).any():
            return
        poly = clip_poly(p, Ww, Hh)
        if fill and len(poly) >= 3:
            dr.polygon(poly, fill=fill)
        if outline:
            for i in range(len(pts_m)):
                seg_m(pts_m[i], pts_m[(i + 1) % len(pts_m)], outline, width)

    ang = np.linspace(0, 2 * np.pi, 73)[:-1]
    for op in ops:
        k = op[0]
        if k == "line":
            _, p0, p1, rgb, a, w = op
            seg_m(p0, p1, tuple(rgb) + (A(a),), max(2, int(w * s)))
        elif k == "ring":
            _, c, R, rgb, fa, la, w = op
            poly_m(np.c_[c[0] + R * np.cos(ang), c[1] + R * np.sin(ang)], tuple(rgb) + (A(fa),) if fa > 0 else None, tuple(rgb) + (A(la),) if la > 0 else None, max(2, int(w * s)))
        elif k in ("rect", "poly"):
            if k == "rect":
                _, (x0, y0, x1, y1), rgb, fa, la = op; pts = [(x0, y0), (x1, y0), (x1, y1), (x0, y1)]
            else:
                _, pts, rgb, fa, la = op
            poly_m(pts, tuple(rgb) + (A(fa),) if fa > 0 else None, tuple(rgb) + (A(la),) if la > 0 else None, max(2, int(4 * s)))
        elif k == "text":
            labels.append(op)

    arr = np.array(ov)                                                       # hide everything behind the players (soft edges)
    mask = np.zeros(arr.shape[:2], np.float32)
    if boxes_f is not None:
        for r in boxes_f.itertuples():
            if pd.isna(r.x1):
                continue
            w = r.x2 - r.x1
            xa, xb = int(max(r.x1 + w * 0.15, 0)), int(max(r.x2 - w * 0.15, 0))
            ya, yb = int(max(r.y1, 0)), int(max(r.y2 - (r.y2 - r.y1) * 0.08, 0))
            mask[ya:yb, xa:xb] = 1.0
        mask = np.clip(cv2.GaussianBlur(mask, (0, 0), max(2.0, 3.5 * s)) * 1.6, 0, 1)
        arr[..., 3] = (arr[..., 3].astype(np.float32) * (1 - mask)).astype(np.uint8)
    out = PILImage.alpha_composite(pil, PILImage.fromarray(arr, "RGBA"))
    if labels:
        d2 = ImageDraw.Draw(out); f_ = _font(max(11, int(15 * s)), True)
        for _, p, txt, rgb in labels:
            q = proj(H, [p])[0]
            if np.isnan(q).any() or not (0 <= q[0] < Ww and 0 <= q[1] < Hh):
                continue
            w_ = d2.textlength(txt, font=f_)
            d2.rounded_rectangle([q[0] - w_ / 2 - 6 * s, q[1] - 12 * s, q[0] + w_ / 2 + 6 * s, q[1] + 12 * s], radius=int(5 * s), fill=tuple(rgb) + (225,))
            d2.text((q[0], q[1]), txt, font=f_, fill=(255, 255, 255, 255), anchor="mm")
    return out

def tags_layer(pil, players, carrier_id, hl):
    """Shirt-number tags above the heads in the team colour, feet ellipses, flipped triangle on the ball carrier, highlighted players."""
    if players is None or len(players) == 0:
        return pil
    Ww, Hh = pil.size; s = Hh / 1080
    nf, lf = _font(max(12, int(15 * s)), True), _font(max(11, int(14 * s)), True)
    ov = PILImage.new("RGBA", pil.size, (0, 0, 0, 0)); dr = ImageDraw.Draw(ov)
    items = [r for r in players.itertuples() if not (pd.isna(r.x1) or pd.isna(r.foot_px_x))]
    for r in sorted(items, key=lambda r: r.foot_px_y):                       # far -> near: nearer tags end up on top
        is_ref = (r.role == "referee")
        col = (110, 110, 110) if is_ref else TEAM_RGB.get(int(r.team), (128, 128, 128)) if pd.notna(r.team) else (128, 128, 128)
        fx, fy, w = int(r.foot_px_x), int(r.foot_px_y), max(int(r.x2 - r.x1), 16)
        dr.ellipse([fx - int(w * 0.45), fy - int(w * 0.14), fx + int(w * 0.45), fy + int(w * 0.14)], outline=col + (225,), width=max(1, int(2 * s)))
        tid = int(r.track_id)
        if tid in hl:                                                         # highlighted by the metric
            hcol, htxt = hl[tid]
            dr.ellipse([fx - int(w * 0.8), fy - int(w * 0.28), fx + int(w * 0.8), fy + int(w * 0.28)], outline=tuple(hcol) + (255,), width=max(3, int(4 * s)))
            if htxt:
                tw = dr.textlength(htxt, font=lf); yy = fy + int(w * 0.34)
                dr.rounded_rectangle([fx - tw / 2 - 5 * s, yy, fx + tw / 2 + 5 * s, yy + 20 * s], radius=int(5 * s), fill=tuple(hcol) + (230,))
                dr.text((fx, yy + 10 * s), htxt, font=lf, fill=(255, 255, 255, 255), anchor="mm")
        if is_ref:
            continue
        num = str(int(r.number)) if pd.notna(r.number) else "?"
        nw = dr.textlength(num, font=nf); lh = int(nf.size + 4 * s); lw = int(nw + 10 * s)
        cx, top = int((r.x1 + r.x2) / 2), int(r.y1)
        x0, y0 = int(np.clip(cx - lw // 2, 2, Ww - lw - 2)), max(top - lh - int(6 * s), 2)
        is_car = carrier_id is not None and tid == carrier_id
        dr.polygon([(x0 + lw // 2 - 4 * s, y0 + lh), (x0 + lw // 2 + 4 * s, y0 + lh), (x0 + lw // 2, y0 + lh + 4 * s)], fill=col + (232,))
        dr.rounded_rectangle([x0, y0, x0 + lw, y0 + lh], radius=int(lh * 0.28), fill=col + (232,),
                             outline=(255, 255, 255, 240) if is_car else (255, 255, 255, 120), width=2 if is_car else 1)
        dr.text((x0 + lw // 2, y0 + lh // 2), num, font=nf, fill=(255, 255, 255, 255), anchor="mm")
        if is_car:
            mx, ty = x0 + lw // 2, y0 - int(3 * s)
            dr.polygon([(mx - 7 * s, ty - 10 * s), (mx + 7 * s, ty - 10 * s), (mx, ty)], fill=(255, 255, 255, 255), outline=(0, 0, 0, 255))
    return PILImage.alpha_composite(pil, ov)

def hud_layer(pil, title, ex, spec, ctx):
    """Top-left panel (metric, example, live numbers), top-right chip (state), bottom-left footer (clock and orientation)."""
    Ww, Hh = pil.size; s = Hh / 1080
    ov = PILImage.new("RGBA", pil.size, (0, 0, 0, 0)); dr = ImageDraw.Draw(ov)
    f_t, f_c, f_l = _font(max(13, int(21 * s)), True), _font(max(11, int(15 * s)), False), _font(max(12, int(17 * s)), False)
    rows = [(title, f_t, (255, 255, 255)), (ex["caption"], f_c, (150, 220, 255))] + [(t, f_l, (255, 255, 255)) for t in spec.get("hud", [])]
    pad = 10 * s; wmax = max(dr.textlength(t, font=f) for t, f, _ in rows)
    hs = [f.size * 1.35 for _, f, _ in rows]
    x0, y0 = 14 * s, 14 * s
    dr.rounded_rectangle([x0, y0, x0 + wmax + 2 * pad, y0 + sum(hs) + 2 * pad], radius=int(8 * s), fill=(15, 15, 15, 165))
    y = y0 + pad
    for (t, f, col), h in zip(rows, hs):
        dr.text((x0 + pad, y), t, font=f, fill=col + (255,)); y += h
    if spec.get("chip"):
        txt, rgb = spec["chip"]; f2 = _font(max(13, int(20 * s)), True); tw = dr.textlength(txt, font=f2)
        dr.rounded_rectangle([Ww - tw - 40 * s, 14 * s, Ww - 14 * s, 14 * s + 38 * s], radius=int(9 * s), fill=tuple(rgb) + (235,))
        dr.text((Ww - 27 * s - tw / 2, 14 * s + 19 * s), txt, font=f2, fill=(255, 255, 255, 255), anchor="mm")
    foot = f"match clock {mmss_clock(ctx.f)}   frame {ctx.f}   {name(LTR_TEAM)} attacks left \u2192 right, {name(RTL_TEAM)} right \u2192 left (minimap)"
    f3 = _font(max(10, int(14 * s)), False); fw = dr.textlength(foot, font=f3)
    dr.rounded_rectangle([14 * s, Hh - 38 * s, 14 * s + fw + 20 * s, Hh - 12 * s], radius=int(6 * s), fill=(15, 15, 15, 150))
    dr.text((24 * s, Hh - 25 * s), foot, font=f3, fill=(235, 235, 235, 255), anchor="lm")
    return PILImage.alpha_composite(pil, ov)

def draw_minimap(img, ctx, spec):
    """Bottom-right pitch map in the figure orientation (Barca left -> right): all players, the pressed target and the metric's ops."""
    Hh, Ww = img.shape[:2]; s = Hh / 1080
    mw, mh = int(340 * s), int(220 * s); pad = max(6, int(8 * s))
    mm = np.full((mh, mw, 3), (48, 108, 52), np.uint8)
    sx, sy = (mw - 2 * pad) / PITCH_LENGTH, (mh - 2 * pad) / PITCH_WIDTH
    def P(x, y):
        dx, dy = to_display(x, y); return (int(round(pad + float(dx) * sx)), int(round(mh - pad - float(dy) * sy)))
    bgr = lambda rgb: (int(rgb[2]), int(rgb[1]), int(rgb[0]))
    wl = (235, 235, 235)
    cv2.rectangle(mm, P(0, 0), P(PITCH_LENGTH, PITCH_WIDTH), wl, 1)
    cv2.line(mm, P(PITCH_LENGTH / 2, 0), P(PITCH_LENGTH / 2, PITCH_WIDTH), wl, 1)
    cv2.circle(mm, P(PITCH_LENGTH / 2, PITCH_WIDTH / 2), int(9.15 * sx), wl, 1)
    for gx, sg in ((0, 1), (PITCH_LENGTH, -1)):
        cv2.rectangle(mm, P(gx, (PITCH_WIDTH - 40.3) / 2), P(gx + sg * 16.5, (PITCH_WIDTH + 40.3) / 2), wl, 1)
    ang = np.linspace(0, 2 * np.pi, 49)[:-1]
    for op in spec.get("ops", []):
        k = op[0]
        if k == "line":
            cv2.line(mm, P(*op[1]), P(*op[2]), bgr(op[3]), max(1, int(op[5] * s * 0.7)), cv2.LINE_AA)
        elif k in ("ring", "rect", "poly"):
            if k == "ring":
                _, c, R, rgb, fa, la, w = op; pts = [P(c[0] + R * np.cos(a), c[1] + R * np.sin(a)) for a in ang]
            elif k == "rect":
                _, (x0, y0, x1, y1), rgb, fa, la = op; pts = [P(x0, y0), P(x1, y0), P(x1, y1), P(x0, y1)]
            else:
                _, pp_, rgb, fa, la = op; pts = [P(*p) for p in pp_]
            pts = np.array(pts, np.int32)
            if fa > 0:
                layer = mm.copy(); cv2.fillPoly(layer, [pts], bgr(rgb)); mm = cv2.addWeighted(layer, min(fa * 2.2, 0.9), mm, 1 - min(fa * 2.2, 0.9), 0)
            if la > 0:
                cv2.polylines(mm, [pts], True, bgr(rgb), 1, cv2.LINE_AA)
        elif k == "text":
            cv2.putText(mm, op[2], P(*op[1]), cv2.FONT_HERSHEY_SIMPLEX, max(0.3, 0.38 * s), bgr(op[3]), 1, cv2.LINE_AA)
    hl = spec.get("hl", {})
    for r in ctx.pos.itertuples():
        col = bgr(TEAM_RGB.get(int(r.team), (128, 128, 128))); rad = max(2, int((2.4 if r.role == "player" else 1.8) * 2 * s))
        cv2.circle(mm, P(r.x, r.y), rad, col, -1, cv2.LINE_AA)
        if ctx.carrier is not None and int(r.track_id) == ctx.carrier:
            cv2.circle(mm, P(r.x, r.y), rad + 2, (255, 255, 255), 1, cv2.LINE_AA)
        if int(r.track_id) in hl:
            cv2.circle(mm, P(r.x, r.y), rad + 3, bgr(hl[int(r.track_id)][0]), 2, cv2.LINE_AA)
    if ctx.tgt is not None:
        cv2.circle(mm, P(*ctx.tgt), max(2, int(3 * s)), (0, 220, 255), -1, cv2.LINE_AA)
    fs = max(0.3, 0.4 * s)
    cv2.putText(mm, f"{name(LTR_TEAM)} attacks ->", (pad, mh - 2), cv2.FONT_HERSHEY_SIMPLEX, fs, bgr(TEAM_RGB[LTR_TEAM]), 1, cv2.LINE_AA)
    tw = cv2.getTextSize(f"<- {name(RTL_TEAM)} attacks", cv2.FONT_HERSHEY_SIMPLEX, fs, 1)[0][0]
    cv2.putText(mm, f"<- {name(RTL_TEAM)} attacks", (mw - pad - tw, 11), cv2.FONT_HERSHEY_SIMPLEX, fs, bgr(TEAM_RGB[RTL_TEAM]), 1, cv2.LINE_AA)
    x0, y0 = Ww - mw - int(18 * s), Hh - mh - int(48 * s)
    roi = img[y0:y0 + mh, x0:x0 + mw]
    img[y0:y0 + mh, x0:x0 + mw] = cv2.addWeighted(mm, 0.9, roi, 0.1, 0)

# ------------------------------------------------------------------ one example = one annotated still frame (PNG, full resolution). No video, no GIF.
def render_example(folder, k, ex):
    key = int(ex["key"])
    lo, hi = max(key - 6, 0), min(key + 6, MAX_FRAME)           # a few neighbouring frames are read only to smooth the camera calibration of the key frame
    rng = np.arange(lo, hi + 1)
    calib = read_calib(lo, hi)
    Hs = smooth_homographies(fit_homographies(calib), rng) if len(calib) else {}
    boxes = smooth_boxes(calib) if len(calib) else calib
    px_f = boxes[boxes["frame_idx"] == key] if len(calib) else None
    if px_f is not None and px_f.empty:
        px_f = None
    bp = ball_px()
    cap = cv2.VideoCapture(str(VERIFY["video_path"]))
    assert cap.isOpened(), f"cannot open {VERIFY['video_path']}"
    cap.set(cv2.CAP_PROP_POS_FRAMES, key + VERIFY["frame_offset"])
    ok, frame = cap.read(); cap.release()
    assert ok, f"cannot read video frame {key + VERIFY['frame_offset']}"
    ctx = make_ctx(key, bp)
    try:
        spec = ex["builder"](ctx, ex) or {}
    except Exception:
        spec = dict(hud=["overlay error, see the notebook output"])
        print(f"  overlay error at frame {key}:\n{traceback.format_exc()}")
    pil = PILImage.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)).convert("RGBA")
    pil = ground_layer(pil, Hs.get(key), spec.get("ops", []), px_f)
    pil = tags_layer(pil, px_f, ctx.carrier, spec.get("hl", {}))
    pil = hud_layer(pil, ex["title"], ex, spec, ctx)
    img = cv2.cvtColor(np.array(pil.convert("RGB")), cv2.COLOR_RGB2BGR)
    b = bp.loc[key] if key in bp.index else None
    draw_ball(img, (b.ball_px_x, b.ball_px_y) if b is not None and pd.notna(b.ball_px_x) else None, low_conf=(b is not None and b.ball_source == "smoothed"))
    draw_minimap(img, ctx, spec)
    png = folder / f"example{k}_frame{key}.png"
    cv2.imwrite(str(png), img)
    return png

def verify_metric(key, title, examples, builder):
    """Renders every example of a metric as ONE annotated photo frame (no collage, no video, no GIF)."""
    if not VERIFY_READY:
        print("real-frame check skipped:", "; ".join(VERIFY_PROBLEMS)); return
    if ONLY is not None and key not in ONLY:
        print(f"{key}: skipped (ONLY = {ONLY})"); return
    examples = [e for e in examples if e is not None]
    if not examples:
        print(f"{title}: no suitable example found in this match"); return
    folder = VERIFY_DIR / key; folder.mkdir(parents=True, exist_ok=True)
    print(f"{title}: {len(examples)} real frame(s) -> {folder}")
    for k, ex in enumerate(examples, 1):
        t0 = time.time()
        png = render_example(folder, k, dict(ex, builder=builder, title=title))
        display(Markdown(f"**Frame {k}/{len(examples)}**: {ex['caption']}  \n`frame {ex['key']} ({mmss_clock(ex['key'])})`  ({time.time() - t0:.0f} s)"))
        display(IPImage(filename=str(png), width=VERIFY["still_width"]))

# ------------------------------------------------------------------ status
print("real-frame verification:", "READY" if VERIFY_READY else "NOT READY")
for p_ in VERIFY_PROBLEMS:
    print("  -", p_)
if VERIFY_READY:
    _cap = cv2.VideoCapture(str(VERIFY["video_path"]))
    print(f"  video: {int(_cap.get(cv2.CAP_PROP_FRAME_WIDTH))}x{int(_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))}, {_cap.get(cv2.CAP_PROP_FPS):.2f} fps, "
          f"{int(_cap.get(cv2.CAP_PROP_FRAME_COUNT)):,} frames | tracking exports frames up to {MAX_FRAME:,} ({MAX_FRAME / FPS / 60:.1f} min)")
    _cap.release()
    print("  outputs go to:", VERIFY_DIR)


## 1. Average defensive line depth per team
Mean distance from own goal of the **last three outfield defenders**, measured only in frames where the team is out of possession (its real defensive shape, not inflated by attacking-phase positioning). Higher number = higher line.

In [ ]:
lf = line_frames().dropna(subset=["last3"])
g = lf.groupby("team")["last3"]
t1 = pd.DataFrame({"frames": g.size(), "seconds": (g.size() / FPS).round(0), "avg_line_depth_m": g.mean().round(1),
                   "avg_line_depth_pct_of_pitch": (100 * g.mean() / PITCH_LENGTH).round(1)})
t1.index = [name(t) for t in t1.index]
display(t1)

A, B = TEAMS[:2]
va, vb = lf.loc[lf.team == A, "last3"].mean(), lf.loc[lf.team == B, "last3"].mean()
hi, lo = (A, B) if va >= vb else (B, A)
print("\nSUMMARY")
print(f"  {name(A)}'s back three sit {va:.1f} m from their own goal on average, {name(B)}'s {vb:.1f} m.")
print(f"  {name(hi)} defend {abs(va - vb):.1f} m higher up the pitch than {name(lo)} (out-of-possession frames only).")

#### Real frames: metric 1
Three real frames: a typical back-three line of each team and the highest line of the match. White line = the team's match average, coloured line = this frame's back three, the three players counted are marked L1 to L3.

In [ ]:
# Real-frame check, metric 1: the average depth of the back line (last three outfield defenders) on three real frames.
if VERIFY_READY:
    lf = line_frames().dropna(subset=["last3"])
    avg3 = lf.groupby("team")["last3"].mean().to_dict()
    lf = lf.merge(fzs()[["frame_idx", "ok_clip"]], on="frame_idx")
    lf = lf[lf["ok_clip"]].assign(dev=lambda d: (d["last3"] - d["team"].map(avg3)).abs())

    def typical_frame(team):
        c = lf[lf["team"] == team]
        return None if c.empty else c.nsmallest(1, "dev").iloc[0]

    examples = []
    for team in FIG_TEAMS:
        r = typical_frame(team)
        if r is not None:
            examples.append(make_example(r["frame_idx"], f"typical frame: {name(team)}'s back three at {r['last3']:.1f} m (its match average is {avg3[team]:.1f} m)", team=team))
    if len(lf):
        r = lf.nlargest(1, "last3").iloc[0]
        examples.append(make_example(r["frame_idx"], f"highest line of the match: {name(int(r['team']))} at {r['last3']:.1f} m", team=int(r["team"])))

    def builder(ctx, ex):
        di = defenders_sorted(ctx)
        if di is None:
            return dict(hud=["(this frame is not in the export: the team is not out of possession here)"])
        t, d = di
        last3, last1 = d["dist"].iloc[:3].mean(), d["dist"].iloc[0]
        col = TEAM_RGB[t]
        ops = [("line", (raw_x(t, avg3[t]), 0), (raw_x(t, avg3[t]), PITCH_WIDTH), (255, 255, 255), 0.8, 3),                  # the team's match average
               ("line", (raw_x(t, last3), 0), (raw_x(t, last3), PITCH_WIDTH), col, 0.95, 6),                                  # the back three, this frame
               ("text", (raw_x(t, last3), PITCH_WIDTH * 0.08), f"{last3:.1f} m", col)]
        hl = {int(r.track_id): (col, f"L{i + 1}") for i, r in enumerate(d.iloc[:3].itertuples())}
        hud = [f"{name(t)} out of possession: back three at {last3:.1f} m from their own goal (mean of the 3 deepest outfield players)",
               f"match average {avg3[t]:.1f} m (white line)  |  this frame {last3 - avg3[t]:+.1f} m  |  last defender alone: {last1:.1f} m"]
        return dict(ops=ops, hl=hl, hud=hud, chip=(f"{name(t)} DEFENDING", col))

    verify_metric("m01_line_depth", "Metric 1 | average defensive line depth", dedupe(examples), builder)

else:
    print("real-frame check skipped:", "; ".join(VERIFY_PROBLEMS))


## 2. Pressing intensity
Share of a team's out-of-possession time during which it is pressing the ball carrier (the pipeline's sustained `under_pressure` flag).
`pressing intensity % = pressing frames / out-of-possession frames`. Pressure in the team's own third is not counted as pressing (see `EXCLUDE_OWN_PRESS_UP_TO_M`), while the time spent defending there stays in the denominator.

In [ ]:
g = frames.groupby("def_team")
t2 = pd.DataFrame({"out_of_possession_frames": g.size(), "pressing_frames": g["under_pressure"].sum().astype(int)})
t2["out_of_possession_sec"] = (t2["out_of_possession_frames"] / FPS).round(1)
t2["pressing_sec"] = (t2["pressing_frames"] / FPS).round(1)
t2["pressing_intensity_pct"] = (100 * t2["pressing_frames"] / t2["out_of_possession_frames"]).round(1)
t2["avg_pressers_when_pressing"] = frames[frames["under_pressure"]].groupby("def_team")["n_pressing"].mean().round(2)
t2.index = [name(t) for t in t2.index]
display(t2)

A, B = TEAMS[:2]
ia, ib = t2.loc[name(A), "pressing_intensity_pct"], t2.loc[name(B), "pressing_intensity_pct"]
hi, lo = (A, B) if ia >= ib else (B, A)
print("\nSUMMARY")
print(f"  {name(A)} pressed the ball carrier for {ia:.1f}% of their out-of-possession time, {name(B)} for {ib:.1f}%.")
print(f"  {name(hi)} pressed more, by {abs(ia - ib):.1f} percentage points.")

#### Real frames: metric 2
Two pressing frames (one per team) and one frame where a team defends without pressing. Rings show the 5 m and 2 m radius around the ball carrier; lines go to the defenders whose press flag is on.

In [ ]:
# Real-frame check, metric 2: pressing intensity. Two pressing frames (one per team) and one frame where the team defends without pressing.
if VERIFY_READY:
    g = frames.groupby("def_team")["under_pressure"]
    intensity = (100 * g.mean()).to_dict()
    examples = []
    for team in FIG_TEAMS:
        f = pick_frame(True, team=team)
        if f is not None:
            examples.append(make_example(f, f"{name(team)} pressing: counts toward its {intensity[team]:.1f}% pressing intensity", team=team,
                                         hud=[f"match pressing intensity: {name(team)} presses {intensity[team]:.1f}% of its out-of-possession time"]))
    team_np = max(TEAMS, key=lambda t: 0 if pick_frame(False, team=t) is None else 1)
    f = pick_frame(False, team=team_np)
    if f is not None:
        examples.append(make_example(f, f"{name(team_np)} out of possession but NOT pressing: counts in the denominator only", team=team_np,
                                     hud=[f"match pressing intensity: {name(team_np)} presses {intensity[team_np]:.1f}% of its out-of-possession time"]))
    verify_metric("m02_pressing_intensity", "Metric 2 | pressing intensity", dedupe(examples), b_press)

else:
    print("real-frame check skipped:", "; ".join(VERIFY_PROBLEMS))


## 3. Pressure by pitch zone (middle and attacking third)
Pressure frames are placed by **where the pressed ball carrier is**, in the pressing team's own attacking frame. Only the middle and attacking third count: pressure in the team's own third is not counted as pressing (the number of frames dropped for that is printed under the table).

In [ ]:
pf_ = frames[frames["under_pressure"]].copy()
pf_["press_x"], pf_["press_y"] = team_frame(pf_["x"], pf_["y"], pf_["def_team"])
pf_["zone"] = zone_of(pf_["press_x"]).to_numpy()
pf_ = pf_.dropna(subset=["zone"])

cnt = pf_.groupby(["def_team", "zone"], observed=False).size().unstack(fill_value=0).reindex(columns=ZONES, fill_value=0)
rows = []
for team in TEAMS:
    c = cnt.loc[team] if team in cnt.index else pd.Series(0, index=ZONES)
    tot_all, tot_ma = int(c.sum()), int(c["middle_third"] + c["attacking_third"])
    for z in ["middle_third", "attacking_third"]:
        rows.append(dict(team=name(team), zone=z, pressure_frames=int(c[z]), seconds=round(c[z] / FPS, 1),
                         pct_of_middle_plus_attacking=round(100 * c[z] / tot_ma, 1) if tot_ma else np.nan,
                         pct_of_all_pressure=round(100 * c[z] / tot_all, 1) if tot_all else np.nan))
t3 = pd.DataFrame(rows)
display(t3.set_index(["team", "zone"]))
_ex = (frames["under_pressure_raw"] & ~frames["under_pressure"]).groupby(frames["def_team"]).sum()
print("raw pressure frames NOT counted because the ball was in the team's own third:", {name(t): int(_ex.get(t, 0)) for t in TEAMS})

print("\nSUMMARY")
for team in TEAMS:
    r = t3[t3.team == name(team)].set_index("zone")
    print(f"  {name(team)}: {r.loc['attacking_third', 'pressure_frames']} attacking-third and {r.loc['middle_third', 'pressure_frames']} middle-third pressure frames "
          f"-> {r.loc['attacking_third', 'pct_of_middle_plus_attacking']:.1f}% of their middle+attacking pressure came in the attacking third"
          + ("" if EXCLUDE_OWN_PRESS_UP_TO_M is not None and EXCLUDE_OWN_PRESS_UP_TO_M >= THIRD_1_M else f" ({r.loc['attacking_third', 'pct_of_all_pressure']:.1f}% of all their pressure)") + ".")

#### Real frames: metric 3
Pressing frames in the attacking and middle third. The third where the ball is while the team presses is shaded; the panel gives the team's counts per zone.

In [ ]:
# Real-frame check, metric 3: pressure by pitch zone. The third where the ball is while the team presses is shaded.
if VERIFY_READY:
    fz = frames_zoned(); pz = fz[fz["under_pressure"]]
    cnt = pz.groupby(["def_team", "zone"]).size().unstack(fill_value=0).reindex(columns=PRESS_ZONES, fill_value=0)
    def line_for(team):
        c = cnt.loc[team] if team in cnt.index else pd.Series(0, index=PRESS_ZONES)
        tot = int(c.sum()); return f"{name(team)} pressure frames: " + ", ".join(f"{z.replace('_third', '')} {int(c[z])} ({100 * c[z] / tot:.0f}%)" for z in PRESS_ZONES) if tot else ""
    examples = []
    for team, zone in [(LTR_TEAM, "attacking_third"), (RTL_TEAM, "attacking_third"), (LTR_TEAM, "middle_third"), (RTL_TEAM, "middle_third")]:
        f = pick_frame(True, team=team, extra=lambda z, zone=zone: z["zone"] == zone)
        if f is not None:
            examples.append(make_example(f, f"{name(team)} pressing in the {zone.replace('_', ' ')}", team=team, shade="zone", hud=[line_for(team)]))
    verify_metric("m03_pressure_zones", "Metric 3 | pressure by pitch zone", dedupe(examples)[:3], b_press)

else:
    print("real-frame check skipped:", "; ".join(VERIFY_PROBLEMS))


## 4. Pressure heatmap
Where each team's pressure is concentrated: position of the pressed ball carrier in every pressure frame, **in the pressing team's attacking frame** (both teams attack left to right, own goal on the left). Each map is normalised to its own team's total, so it shows *where* pressure concentrates, not how much there is.

In [ ]:
pp = frames[frames["under_pressure"]].dropna(subset=["x", "y"]).copy()
tol = PRM["BOUNDS_TOL_M"]
inb = pp["x"].between(-tol, PITCH_LENGTH + tol) & pp["y"].between(-tol, PITCH_WIDTH + tol)
print(f"dropped {int((~inb).sum())} / {len(pp)} pressure points outside the pitch (+/- {tol} m)")
pp = pp[inb].copy()
pp["press_x"], pp["press_y"] = team_frame(pp["x"], pp["y"], pp["def_team"])        # in the pressing team's own attacking frame (for the table)
pp["dx"], pp["dy"] = to_display(pp["x"], pp["y"])                                   # in the figure frame: Barca attacks left -> right

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
for ax, team in zip(axes, FIG_TEAMS):
    sub = pp[pp["def_team"] == team]
    draw_pitch(ax)
    if EXCLUDE_OWN_PRESS_UP_TO_M is not None and EXCLUDE_OWN_PRESS_UP_TO_M >= THIRD_1_M:       # own third: not counted as pressing
        x0, x1 = (0, THIRD_1_M) if team == LTR_TEAM else (PITCH_LENGTH - THIRD_1_M, PITCH_LENGTH)
        ax.add_patch(patches.Rectangle((x0, 0), x1 - x0, PITCH_WIDTH, facecolor="#bbbbbb", alpha=0.35, hatch="//", edgecolor="none"))
        ax.text((x0 + x1) / 2, PITCH_WIDTH + 1.5, "own third: not counted", ha="center", fontsize=8, color="#555555")
    if len(sub):
        w = np.full(len(sub), 100.0 / len(sub))
        hb = ax.hexbin(sub["dx"], sub["dy"], C=w, reduce_C_function=np.sum, gridsize=PRM["HEATMAP_GRIDSIZE"],
                       cmap=TEAM_CMAP[team], mincnt=1, extent=(0, PITCH_LENGTH, 0, PITCH_WIDTH), alpha=0.85)
        plt.colorbar(hb, ax=ax, shrink=0.7).set_label("% of the team's own pressure frames", fontsize=9)
    ax.set_title(f"{name(team)} pressure density   " + ("attacks \u2192" if team == LTR_TEAM else "\u2190 attacks"), fontsize=12)
plt.tight_layout(); plt.show()

rows = []
for team in TEAMS:
    s = pp[pp["def_team"] == team]
    ch = channel_of(s["press_y"]).astype(object).map(CH_LABEL)
    chp = ch.value_counts(normalize=True).reindex(["left", "centre", "right"]).fillna(0) * 100
    rows.append(dict(team=name(team), pressure_frames=len(s), centroid_x_m=round(s["press_x"].mean(), 1), centroid_y_m=round(s["press_y"].mean(), 1),
                     pct_in_opponent_half=round(100 * (s["press_x"] >= PITCH_LENGTH / 2).mean(), 1),
                     pct_left=round(chp["left"], 1), pct_centre=round(chp["centre"], 1), pct_right=round(chp["right"], 1)))
t4 = pd.DataFrame(rows).set_index("team")
display(t4)
print("(centroid_x_m = metres from the pressing team's own goal; centroid_y_m and left / right are in its own attacking direction)")

print("SUMMARY")
for team in TEAMS:
    r = t4.loc[name(team)]
    side = r[["pct_left", "pct_centre", "pct_right"]].astype(float).idxmax().replace("pct_", "")
    print(f"  {name(team)}: pressure centred {r['centroid_x_m']:.0f} m from their own goal, {r['pct_in_opponent_half']:.0f}% of it in the opponent half; "
          f"most pressure in the {side} channel (left / right as seen by the pressing team, looking at the opponent's goal).")


#### Real frames: metric 4
Pressing moments inside the hottest 10 m cell of each team's heatmap (shaded).

In [ ]:
# Real-frame check, metric 4: the heatmap. The hottest 10 m cell of each team (in its own attacking frame) is shaded; frames are pressing moments inside it.
if VERIFY_READY:
    z = fzs(); pz = z[z["under_pressure"]].dropna(subset=["press_x", "press_y"])
    xs, ys = np.linspace(0, PITCH_LENGTH, 11), np.linspace(0, PITCH_WIDTH, 7)
    ranked = {}
    for team in TEAMS:
        s = pz[pz["def_team"] == team]
        Hh, _, _ = np.histogram2d(s["press_x"], s["press_y"], bins=[xs, ys])
        order = np.dstack(np.unravel_index(np.argsort(-Hh, axis=None), Hh.shape))[0]
        ranked[team] = [(int(i), int(j), Hh[i, j], 100 * Hh[i, j] / max(Hh.sum(), 1)) for i, j in order]
    def cell_example(team, rank):
        for i, j, n, pct in ranked[team][rank:rank + 6]:
            f = pick_frame(True, team=team, extra=lambda zz, i=i, j=j: zz["press_x"].between(xs[i], xs[i + 1]) & zz["press_y"].between(ys[j], ys[j + 1]))
            if f is not None:
                xr = sorted([raw_x(team, xs[i]), raw_x(team, xs[i + 1])]); yr = sorted([raw_y(team, ys[j]), raw_y(team, ys[j + 1])])
                return make_example(f, f"{name(team)}: hottest-cell #{rank + 1} ({pct:.1f}% of its pressure frames in this 10 m cell)", team=team,
                                    rect=(xr[0], yr[0], xr[1], yr[1]), hud=[f"{name(team)}'s pressure heatmap peaks here: {int(n)} frames = {pct:.1f}% of its pressure"])
        return None
    more = max(TEAMS, key=lambda t: len(pz[pz["def_team"] == t]))
    examples = [cell_example(LTR_TEAM, 0), cell_example(RTL_TEAM, 0), cell_example(more, 1)]
    verify_metric("m04_pressure_heatmap", "Metric 4 | pressure heatmap (hottest cells)", dedupe(examples), b_press)

else:
    print("real-frame check skipped:", "; ".join(VERIFY_PROBLEMS))


## 5. Pressing intensity over time (5-minute bins)
% of a team's out-of-possession frames spent pressing, per 5-minute block of the match clock. A team/bin with fewer than `MIN_BIN_FRAMES` out-of-possession frames is **masked** (shown as NaN and left out of the plot), because one short sequence would swing the average.

In [ ]:
bm, minf = PRM["BIN_MINUTES"], PRM["MIN_BIN_FRAMES"]
fr = frames.copy()
fr["bin"] = (fr["frame_idx"] / FPS // (bm * 60)).astype(int)
agg = fr.groupby(["def_team", "bin"]).agg(n_frames=("under_pressure", "size"), pct=("under_pressure", "mean")).reset_index()
agg["pct"] = 100 * agg["pct"]
agg["masked"] = agg["n_frames"] < minf
agg.loc[agg["masked"], "pct"] = np.nan
agg["bin_label"] = agg["bin"].apply(lambda b: f"{b * bm}-{(b + 1) * bm} min")

pct_tab = agg.pivot(index="bin", columns="def_team", values="pct")
n_tab = agg.pivot(index="bin", columns="def_team", values="n_frames").fillna(0).astype(int)
t5 = pd.DataFrame(index=pct_tab.index)
for team in TEAMS:
    t5[f"{name(team)} pressing %"] = pct_tab[team].round(1) if team in pct_tab else np.nan
    t5[f"{name(team)} frames"] = n_tab[team] if team in n_tab else 0
t5.index = [f"{b * bm}-{(b + 1) * bm} min" for b in t5.index]
display(t5)
print(f"masked team/bin cells (fewer than {minf} out-of-possession frames = {minf / FPS:.0f} s):", int(agg["masked"].sum()))

fig, ax = plt.subplots(figsize=(10, 4.5))
for team in FIG_TEAMS:
    col = TEAM_COLOR[team]
    s = agg[agg["def_team"] == team].sort_values("bin")
    ax.plot(s["bin"], s["pct"], marker="o", linewidth=2, color=col, label=name(team))
ax.set_xticks(sorted(agg["bin"].unique())); ax.set_xticklabels([f"{b * bm}-{(b + 1) * bm}" for b in sorted(agg["bin"].unique())])
ax.set_xlabel("match minute"); ax.set_ylabel("% of out-of-possession time pressing"); ax.grid(alpha=0.3); ax.legend()
ax.set_title("Pressing intensity per 5-minute block (masked bins not drawn)")
plt.tight_layout(); plt.show()

print("SUMMARY")
for team in TEAMS:
    s = agg[(agg["def_team"] == team) & ~agg["masked"]]
    if s.empty:
        print(f"  {name(team)}: no bin with enough frames."); continue
    hi_, lo_ = s.loc[s["pct"].idxmax()], s.loc[s["pct"].idxmin()]
    print(f"  {name(team)}: highest in {hi_['bin_label']} ({hi_['pct']:.1f}%), lowest in {lo_['bin_label']} ({lo_['pct']:.1f}%), "
          f"{len(s)} reliable bins out of {int((agg['def_team'] == team).sum())}.")

#### Real frames: metric 5
A pressing moment from the peak 5-minute bin of each team and from the lowest reliable bin, with the bin's percentage.

In [ ]:
# Real-frame check, metric 5: pressing intensity per 5-minute bin. Frames come from the peak bin of each team and from the lowest reliable bin.
if VERIFY_READY:
    bm, minf = PRM["BIN_MINUTES"], PRM["MIN_BIN_FRAMES"]
    fr_ = frames.assign(bin=(frames["frame_idx"] / FPS // (bm * 60)).astype(int))
    agg = fr_.groupby(["def_team", "bin"]).agg(n=("under_pressure", "size"), pct=("under_pressure", "mean")).reset_index()
    agg["pct"] *= 100; agg = agg[agg["n"] >= minf]
    def bin_example(team, which):
        a = agg[agg["def_team"] == team] if team is not None else agg
        if a.empty:
            return None
        r = a.loc[a["pct"].idxmax() if which == "peak" else a["pct"].idxmin()]
        t, b = int(r["def_team"]), int(r["bin"])
        in_bin = lambda z, b=b: (z["frame_idx"] / FPS // (bm * 60)).astype(int) == b
        f = pick_frame(True, team=t, extra=in_bin) or pick_frame(False, team=t, extra=in_bin)
        if f is None:
            return None
        lab = f"{b * bm}-{(b + 1) * bm} min"
        return make_example(f, f"{name(t)}, {which} bin {lab}: pressing {r['pct']:.1f}% of its out-of-possession frames", team=t,
                            hud=[f"bin {lab}: {name(t)} pressed {r['pct']:.1f}% of {int(r['n'])} out-of-possession frames (bins under {minf} frames are masked)"])
    examples = [bin_example(LTR_TEAM, "peak"), bin_example(RTL_TEAM, "peak"), bin_example(None, "lowest")]
    verify_metric("m05_pressing_over_time", "Metric 5 | pressing intensity over time (5-minute bins)", dedupe(examples), b_press)

else:
    print("real-frame check skipped:", "; ".join(VERIFY_PROBLEMS))


## 6. PPDA (passes per defensive action)
`PPDA = opponent completed passes / defensive actions` (interceptions + tackles) of the defending team, **both counted outside that team's own defensive third** (own third = the first 35 m from its own goal). Zones are always taken in the *defending* team's frame. Lower PPDA = more aggressive, higher-up pressing.
Passes and actions are rebuilt from carrier transitions in `frames`: same-team handover >= 3 m = completed pass, team change >= 3 m = interception, team change < 3 m = tackle, keeper claims excluded.

In [ ]:
res = detect_passes(); passes, actions, info = res["passes"], res["actions"], res["info"]
print(f"detection: {info['carrier_runs']} carrier runs, {info['transitions']} transitions ({info['within_gap']} within {PRM['MAX_RECEPTION_GAP_SEC']} s), "
      f"{info['missing_positions']} skipped for missing positions, {info['gk_claims_excluded']} keeper claims excluded -> "
      f"{len(passes)} pass attempts, {len(actions)} defensive actions\n")

t6 = ppda_table()
display(t6)

A, B = TEAMS[:2]
pa, pb = t6.loc[name(A), "PPDA"], t6.loc[name(B), "PPDA"]
print("\nSUMMARY")
print(f"  PPDA: {name(A)} {pa:.2f}, {name(B)} {pb:.2f} (lower = more aggressive pressing).")
if pd.notna(pa) and pd.notna(pb):
    print(f"  {name(A if pa < pb else B)} press more aggressively by this measure.")
small = [n for n in t6.index if t6.loc[n, "defensive_actions"] < 10]
if small:
    print(f"  CAUTION: fewer than 10 defensive actions for {small}; PPDA is unstable at that sample size.")

#### Real frames: metric 6
An interception, a tackle (or a second interception if there is no tackle) and a pass allowed, each outside the defending team's own third. The pass line runs from the release point to the receiver.

In [ ]:
# Real-frame check, metric 6: PPDA. A defensive action (interception / tackle) and a pass allowed, both outside the defending team's own third.
if VERIFY_READY:
    res = detect_passes(); passes, actions = res["passes"], res["actions"]; t6 = ppda_table()
    pk = lambda df, team_col, xcol, ycol, team: team_frame(df[xcol], df[ycol], np.full(len(df), team))[0] >= THIRD_1_M     # outside the defending team's own third
    acts, allowed = [], []
    for team in TEAMS:
        a = actions[actions["defending_team"] == team]; acts.append(a[pk(a, "defending_team", "x", "y", team)])
        p = passes[(passes["outcome"] == "completed") & (passes["passer_team"] == 1 - team)]; allowed.append(p[pk(p, "passer_team", "release_x", "release_y", team)].assign(defending_team=team))
    acts, allowed = pd.concat(acts).sort_values("frame"), pd.concat(allowed).sort_values("release_frame")
    def mid(df):
        return None if df.empty else df.iloc[len(df) // 2]
    inter = mid(acts[acts["action_type"] == "interception"]); tack = mid(acts[acts["action_type"] == "tackle"])
    if tack is None:
        rest = acts[(acts["action_type"] == "interception") & (acts["frame"] != (inter["frame"] if inter is not None else -1))]
        tack, tack_kind = mid(rest), "interception"
    else:
        tack_kind = "tackle"
    allow = mid(allowed)
    def ppda_line(team):
        r = t6.loc[name(team)]; return f"{name(team)} PPDA {r['PPDA']:.2f} = {int(r['passes_allowed'])} passes allowed / {int(r['defensive_actions'])} defensive actions (own third excluded)"
    def action_example(r, kind):
        if r is None:
            return None
        d = int(r["defending_team"])
        return make_example(r["reception_frame"], f"{name(d)} {kind}: counts as a defensive action in the PPDA of {name(d)}", f0=r["frame"] - F(1.0), f1=r["reception_frame"] + F(1.0), max_sec=5,
                            kind=kind, rel=int(r["frame"]), rec=int(r["reception_frame"]), p0=(r["x"], r["y"]), p1=(r["x_end"], r["y_end"]),
                            passer=int(r["attacker_track_id"]), receiver=int(r["defender_track_id"]), att=int(r["attacking_team"]), dfn=d)
    examples = [action_example(inter, "interception"), action_example(tack, tack_kind)]
    if allow is not None:
        d = int(allow["defending_team"])
        examples.append(make_example(allow["reception_frame"], f"pass allowed to {name(1 - d)}: counts in the numerator of {name(d)}'s PPDA", f0=allow["release_frame"] - F(1.0), f1=allow["reception_frame"] + F(1.0), max_sec=5,
                                     kind="completed", rel=int(allow["release_frame"]), rec=int(allow["reception_frame"]), p0=(allow["release_x"], allow["release_y"]), p1=(allow["receive_x"], allow["receive_y"]),
                                     passer=int(allow["passer_track_id"]), receiver=int(allow["receiver_track_id"]), att=1 - d, dfn=d))

    def builder(ctx, ex):
        kind, f = ex["kind"], ctx.f
        col = OK_RGB if kind == "completed" else BAD_RGB
        hl = {ex["passer"]: (TEAM_RGB[ex["att"]], "passer"), ex["receiver"]: (col, {"interception": "intercepts", "tackle": "wins the ball", "completed": "receives"}[kind])}
        ops = [("line", ex["p0"], ex["p1"], col, 0.95 if f >= ex["rel"] else 0.35, 5)]
        if f < ex["rel"]:
            chip = ("BALL WITH THE PASSER", GREY_RGB)
        elif f < ex["rec"]:
            chip = ("PASS IN FLIGHT", (230, 160, 30))
        else:
            chip = ({"interception": "INTERCEPTION", "tackle": "TACKLE", "completed": "PASS ALLOWED"}[kind], col)
        d = ex["dfn"]; dist = float(np.hypot(ex["p1"][0] - ex["p0"][0], ex["p1"][1] - ex["p0"][1]))
        what = {"interception": "team change at 3 m or more = interception (defensive action)", "tackle": "team change under 3 m = tackle (defensive action)",
                "completed": "same-team handover at 3 m or more = completed pass allowed"}[kind]
        hud = [f"{what}; carrier-to-carrier distance {dist:.1f} m", f"counted outside {name(d)}'s own third", ppda_line(d)]
        return dict(ops=ops, hl=hl, hud=hud, chip=chip)

    verify_metric("m06_ppda", "Metric 6 | PPDA (passes allowed / defensive actions)", dedupe(examples, 3), builder)

else:
    print("real-frame check skipped:", "; ".join(VERIFY_PROBLEMS))


## 7. Time-to-press after losing the ball
For every **turnover** (a possession spell that starts right after the opponent had the ball, `start_type == "turnover"`), the time from the start of the new spell until the team that just lost the ball is first pressing the new carrier, searched within `TTP_WINDOW_SEC` (10 s). Turnovers with no press in the window are counted but have no time, so they are not in the mean / median / min / max.

In [ ]:
tv = turnover_table()

rows = []
for team in TEAMS:
    s = tv[tv["losing_team"] == team]; ttp = s["time_to_press_sec"].dropna()
    rows.append(dict(team_that_lost_ball=name(team), turnovers=len(s), pressed_within_window=int(s["pressed"].sum()),
                     pressed_pct=round(100 * s["pressed"].mean(), 1) if len(s) else np.nan,
                     mean_sec=round(ttp.mean(), 2) if len(ttp) else np.nan, median_sec=round(ttp.median(), 2) if len(ttp) else np.nan,
                     min_sec=round(ttp.min(), 2) if len(ttp) else np.nan, max_sec=round(ttp.max(), 2) if len(ttp) else np.nan,
                     not_pressed=int((~s["pressed"]).sum()), of_which_new_spell_shorter_than_window=int(s["ended_before_window"].sum())))
t7 = pd.DataFrame(rows).set_index("team_that_lost_ball")
display(t7)
print(f"window = {PRM['TTP_WINDOW_SEC']:.0f} s, grace = {PRM['TTP_GRACE_SEC']:.1f} s, {len(tv)} turnovers in total")

print("\nSUMMARY")
for team in TEAMS:
    r = t7.loc[name(team)]
    if r["pressed_within_window"] == 0:
        print(f"  {name(team)}: no press detected within {PRM['TTP_WINDOW_SEC']:.0f} s after any of their {int(r['turnovers'])} turnovers."); continue
    print(f"  {name(team)}: after losing the ball {int(r['turnovers'])} times they pressed within {PRM['TTP_WINDOW_SEC']:.0f} s on {int(r['pressed_within_window'])} "
          f"({r['pressed_pct']:.0f}%), median {r['median_sec']:.2f} s (mean {r['mean_sec']:.2f}, min {r['min_sec']:.2f}, max {r['max_sec']:.2f}).")
if PRM["TTP_GRACE_SEC"] == 0:
    print("  NOTE: with TTP_GRACE_SEC = 0 the shortest times can just be the turnover contest itself (the player who lost the ball is still beside the new carrier). "
          "Set TTP_GRACE_SEC = 1.0 to ignore the first second.")

#### Real frames: metric 7
Three real turnovers (median, fastest and slowest press). The clock starts when the new team wins the ball and stops at the first press.

In [ ]:
# Real-frame check, metric 7: time-to-press. Three real turnovers (median, fastest, slowest); the clock runs from the moment the new team wins the ball.
if VERIFY_READY:
    tv = turnover_table(); tp = tv[tv["pressed"]].copy()
    stat = tp.groupby("losing_team")["time_to_press_sec"].agg(["median", "count"]).to_dict("index")
    examples = []
    if len(tp):
        med = tp["time_to_press_sec"].median()
        picks = [("closest to the median", tp.assign(d=(tp["time_to_press_sec"] - med).abs()).nsmallest(1, "d").iloc[0]),
                 ("fastest press", tp.nsmallest(1, "time_to_press_sec").iloc[0]), ("slowest press", tp.nlargest(1, "time_to_press_sec").iloc[0])]
        for label, r in picks:
            lose, gain = int(r["losing_team"]), int(r["team"])
            examples.append(make_example(r["first_press_frame"], f"{label}: {name(gain)} win the ball, {name(lose)} first press after {r['time_to_press_sec']:.2f} s", f0=r["start_frame"] - F(1.0),
                                         f1=r["first_press_frame"] + F(1.5), max_sec=9, start=int(r["start_frame"]), first=int(r["first_press_frame"]), tt=float(r["time_to_press_sec"]), loser=lose, gainer=gain))

    def builder(ctx, ex):
        f, start, first, loser, gainer = ctx.f, ex["start"], ex["first"], ex["loser"], ex["gainer"]
        t = (f - start) / FPS
        show = ctx.fr is not None and f >= start and int(ctx.fr["def_team"]) == loser
        ops, hl = press_ops(ctx) if show else ([], {})
        st = stat.get(loser, dict(median=np.nan, count=0))
        hud = [f"turnover: {name(gainer)} win the ball at {mmss_clock(start)}; {name(loser)} must press (window {PRM['TTP_WINDOW_SEC']:.0f} s)",
               f"time since the turnover: {max(t, 0):.2f} s   |   first press at {ex['tt']:.2f} s = this turnover's value in the metric",
               f"{name(loser)}: median {st['median']:.2f} s over {int(st['count'])} pressed turnovers"]
        if f < start:
            chip = (f"{name(loser)} STILL HAVE THE BALL", GREY_RGB)
        elif f < first:
            chip = (f"{name(gainer)} HAVE THE BALL  {t:.2f} s", (230, 160, 30))
        else:
            chip = (f"PRESSING AFTER {ex['tt']:.2f} s", PRESS_RGB)
        return dict(ops=ops, hl=hl, hud=hud, chip=chip)

    verify_metric("m07_time_to_press", "Metric 7 | time-to-press after losing the ball", dedupe(examples, 3), builder)

else:
    print("real-frame check skipped:", "; ".join(VERIFY_PROBLEMS))


## 8. Pass success rate, under pressure vs not
A pass attempt (completed or intercepted; tackles and keeper claims are not passes) counts as **under pressure** if the opponent was pressing (sustained `under_pressure`) at any frame in the `PRESSURE_LOOKBACK_SEC` (1 s) up to the release. Success = completed. 95% Wilson intervals are shown because the counts are small.

**What this measures:** completion among *resolved* passes only. A pass counts as failed only when an opponent takes the ball at 3 m or more from the passer. Duel losses (dispossessed within 3 m) and balls out of play (gap over 2 s) are not in this table, and both are more common under pressure, so a higher rate under pressure does not mean pressure helps. See 8b for the check.

In [ ]:
passes = pass_pressure_table()

rows = []
for team in TEAMS:
    for state, label in [(True, "under pressure"), (False, "not under pressure")]:
        s = passes[(passes["passer_team"] == team) & (passes["under_pressure"] == state)]
        n, k = len(s), int(s["success"].sum()); lo, hi = wilson(k, n)
        rows.append(dict(team=name(team), state=label, passes=n, completed=k, intercepted=n - k,
                         success_rate_pct=round(100 * k / n, 1) if n else np.nan,
                         ci95=f"{100 * lo:.0f}-{100 * hi:.0f}" if n else "n/a"))
t8 = pd.DataFrame(rows).set_index(["team", "state"])
display(t8)

print("\nSUMMARY")
for team in TEAMS:
    u, nu = t8.loc[(name(team), "under pressure")], t8.loc[(name(team), "not under pressure")]
    share = 100 * u["passes"] / max(1, u["passes"] + nu["passes"])
    if u["passes"] and nu["passes"]:
        d = u["success_rate_pct"] - nu["success_rate_pct"]
        print(f"  {name(team)}: {u['success_rate_pct']:.1f}% of passes completed under pressure ({int(u['passes'])} passes) vs {nu['success_rate_pct']:.1f}% "
              f"without ({int(nu['passes'])}), a gap of {d:+.1f} points; {share:.0f}% of their passes were under pressure.")
    else:
        print(f"  {name(team)}: not enough passes in one of the two states to compare.")
print("  Wide intervals mean the difference between the two states may not be reliable. Duel losses and out-of-play balls are NOT counted as failures here (see 8b).")

#### Real frames: metric 8
A pressed completed pass, a pressed intercepted pass and an unpressed completed pass. The one-second lookback window before the release is flagged.

In [ ]:
# Real-frame check, metric 8: pass success under pressure. A pressed completed pass, a pressed intercepted pass and an unpressed completed pass.
if VERIFY_READY:
    P8 = pass_pressure_table()
    rate = {}
    for team in TEAMS:
        for st in (True, False):
            s = P8[(P8["passer_team"] == team) & (P8["under_pressure"] == st)]; rate[(team, st)] = (100 * s["success"].mean() if len(s) else np.nan, len(s))
    def mid(df):
        return None if df.empty else df.iloc[len(df) // 2]
    def pass_example(r, label):
        if r is None:
            return None
        t = int(r["passer_team"]); opp = 1 - t
        return make_example(r["release_frame"], f"{label}: {name(t)} pass, {'under pressure' if r['under_pressure'] else 'not under pressure'}, {r['outcome']}", f0=r["release_frame"] - F(1.2),
                            f1=r["reception_frame"] + F(1.0), max_sec=5, rel=int(r["release_frame"]), rec=int(r["reception_frame"]), p0=(r["release_x"], r["release_y"]), p1=(r["receive_x"], r["receive_y"]),
                            passer=int(r["passer_track_id"]), receiver=int(r["receiver_track_id"]), team=t, opp=opp, pressed=bool(r["under_pressure"]), ok=bool(r["success"]))
    examples = [pass_example(mid(P8[P8["under_pressure"] & P8["success"]]), "pressed + completed"), pass_example(mid(P8[P8["under_pressure"] & ~P8["success"]]), "pressed + intercepted"),
                pass_example(mid(P8[~P8["under_pressure"] & P8["success"]]), "unpressed + completed")]

    def builder(ctx, ex):
        f, t = ctx.f, ex["team"]
        col = OK_RGB if ex["ok"] else BAD_RGB
        ops = [("line", ex["p0"], ex["p1"], col, 0.95 if f >= ex["rel"] else 0.3, 5)]; hl = {ex["passer"]: (TEAM_RGB[t], "passer"), ex["receiver"]: (col, "receiver")}
        pressing_now = ctx.fr is not None and ctx.under and int(ctx.fr["def_team"]) == ex["opp"]
        if pressing_now:
            o2, h2 = press_ops(ctx); ops += o2
            for k_, v_ in h2.items():
                hl.setdefault(k_, v_)
        win = ex["rel"] - F(PRM["PRESSURE_LOOKBACK_SEC"]) <= f <= ex["rel"]
        if f < ex["rel"]:
            chip = (("LOOKBACK WINDOW" if win else "BEFORE THE PASS"), (230, 160, 30) if win else GREY_RGB)
        elif f < ex["rec"]:
            chip = ("PASS IN FLIGHT", (230, 160, 30))
        else:
            chip = ("COMPLETED" if ex["ok"] else "INTERCEPTED", col)
        ru, rn = rate[(t, True)], rate[(t, False)]
        hud = [f"{name(t)} pass: opponent pressing in the {PRM['PRESSURE_LOOKBACK_SEC']:.0f} s before the release = " + ("YES, under pressure" if ex["pressed"] else "NO, not under pressure"),
               f"result: " + ("completed" if ex["ok"] else "intercepted"),
               f"{name(t)} success rate: {ru[0]:.1f}% under pressure ({ru[1]} passes) vs {rn[0]:.1f}% not ({rn[1]} passes)"]
        return dict(ops=ops, hl=hl, hud=hud, chip=chip)

    verify_metric("m08_pass_pressure", "Metric 8 | pass success under pressure vs not", dedupe(examples, 3), builder)

else:
    print("real-frame check skipped:", "; ".join(VERIFY_PROBLEMS))


## 8b. Check on section 8: what the pressed / unpressed passes look like, and retention with duel losses counted
Section 8 can show a *higher* completion rate under pressure because (a) duel losses are not counted as failed passes and (b) pressed carriers play shorter, safer passes. This cell prints both effects:
1. pass length and direction by pressure state;
2. **retention** = completed passes / (all pass attempts + duel losses), where a duel loss is a carrier dispossessed within 3 m of the next carrier.

In [ ]:
res = detect_passes(); passes = res["passes"].copy(); actions = res["actions"]; info = res["info"]
def add_pressure(df, team_col, frame_col):
    df = df.copy(); df["under_pressure"] = False
    for team in TEAMS:
        mk = (df[team_col] == team).to_numpy()
        if mk.any():
            df.loc[mk, "under_pressure"] = pressed_at(team, df.loc[mk, frame_col])
    return df

passes = add_pressure(passes, "passer_team", "release_frame")
passes["success"] = passes["outcome"] == "completed"
duels = add_pressure(actions[actions["action_type"] == "tackle"], "attacking_team", "frame")     # carrier lost the ball in a duel

# 1) pass length and direction (progression in the passer's own attacking frame)
xr = team_frame(passes["release_x"], passes["release_y"], passes["passer_team"])[0]
xc = team_frame(passes["receive_x"], passes["receive_y"], passes["passer_team"])[0]
passes["progression_m"] = xc - xr
rows = []
for team in TEAMS:
    for state, label in [(True, "under pressure"), (False, "not under pressure")]:
        s = passes[(passes["passer_team"] == team) & (passes["under_pressure"] == state)]
        rows.append(dict(team=name(team), state=label, passes=len(s),
                         median_length_m=round(s["dist_m"].median(), 1), mean_length_m=round(s["dist_m"].mean(), 1),
                         pct_forward_3m=round(100 * (s["progression_m"] >= 3).mean(), 1) if len(s) else np.nan,
                         pct_backward_3m=round(100 * (s["progression_m"] <= -3).mean(), 1) if len(s) else np.nan,
                         pct_intercepted=round(100 * (~s["success"]).mean(), 1) if len(s) else np.nan))
print("1) what the passes look like:")
display(pd.DataFrame(rows).set_index(["team", "state"]))

# 2) retention with duel losses counted as failures
rows = []
for team in TEAMS:
    for state, label in [(True, "under pressure"), (False, "not under pressure")]:
        s = passes[(passes["passer_team"] == team) & (passes["under_pressure"] == state)]
        d = duels[(duels["attacking_team"] == team) & (duels["under_pressure"] == state)]
        n, k = len(s) + len(d), int(s["success"].sum()); lo, hi = wilson(k, n)
        rows.append(dict(team=name(team), state=label, passes=len(s), completed=k, intercepted=int((~s["success"]).sum()), lost_in_duel=len(d),
                         pass_completion_pct=round(100 * k / len(s), 1) if len(s) else np.nan,
                         retention_pct=round(100 * k / n, 1) if n else np.nan, ci95=f"{100 * lo:.0f}-{100 * hi:.0f}" if n else "n/a"))
t8b = pd.DataFrame(rows).set_index(["team", "state"])
print("\n2) retention = completed / (passes + duel losses):")
display(t8b)
print(f"balls with a gap over {PRM['MAX_RECEPTION_GAP_SEC']:.0f} s (out of play / long loose ball), not counted in either table: {info['transitions'] - info['within_gap']}")

print("\nSUMMARY")
for team in TEAMS:
    u, nu = t8b.loc[(name(team), "under pressure")], t8b.loc[(name(team), "not under pressure")]
    print(f"  {name(team)}: pass completion {u['pass_completion_pct']:.1f}% pressed vs {nu['pass_completion_pct']:.1f}% unpressed; "
          f"with {int(u['lost_in_duel'])} pressed and {int(nu['lost_in_duel'])} unpressed duel losses counted, retention is {u['retention_pct']:.1f}% vs {nu['retention_pct']:.1f}%.")
print("  If the pressed rate stays above the unpressed one, look at table 1: shorter, more backward passes under pressure explain it.")

## 9. Defensive line height (last-1 and last-3 defenders)
Distance from own goal of the **last outfield defender** and of the **mean of the last three**, per frame, out-of-possession frames only. Outliers (tracking glitches) are removed per team with the 1.5 x IQR rule before the mean / median / std are computed.

In [ ]:
lf = line_frames()
rows = []
for col, label in [("last1", "last-1 defender"), ("last3", "last-3 defenders (mean)")]:
    base = lf.dropna(subset=[col])
    clean = iqr_clean(base, col)
    for team in TEAMS:
        b, c = base[base["team"] == team][col], clean[clean["team"] == team][col]
        rows.append(dict(team=name(team), line=label, mean_m=round(c.mean(), 1), median_m=round(c.median(), 1), std_m=round(c.std(), 1),
                         frames_kept=len(c), outliers_removed=len(b) - len(c)))
t9 = pd.DataFrame(rows).set_index(["line", "team"])
display(t9)

print("\nSUMMARY")
A, B = TEAMS[:2]
for line in ["last-1 defender", "last-3 defenders (mean)"]:
    a, b = t9.loc[(line, name(A))], t9.loc[(line, name(B))]
    print(f"  {line}: {name(A)} {a['mean_m']:.1f} m (median {a['median_m']:.1f}, std {a['std_m']:.1f}) vs {name(B)} {b['mean_m']:.1f} m "
          f"(median {b['median_m']:.1f}, std {b['std_m']:.1f}); {name(A if a['mean_m'] > b['mean_m'] else B)} defend higher.")
print("  A larger std means the line moves up and down the pitch more.")

#### Real frames: metric 9
A typical last-1 / last-3 line, the highest line that is kept, and a frame that the 1.5 x IQR rule removes as an outlier (if the match has one).

In [ ]:
# Real-frame check, metric 9: last-1 and last-3 line height, including a frame whose value is removed as an outlier.
if VERIFY_READY:
    lf = line_frames()
    B = {}
    for col in ("last1", "last3"):
        for team in TEAMS:
            v = lf.loc[lf["team"] == team, col].dropna(); q1, q3 = v.quantile([0.25, 0.75]); i = q3 - q1
            B[(col, team)] = (q1 - PRM["IQR_MULT"] * i, q3 + PRM["IQR_MULT"] * i, v.median())
    lo1, hi1, lo3, hi3 = (lf["team"].map(lambda t: B[("last1", t)][0]), lf["team"].map(lambda t: B[("last1", t)][1]),
                          lf["team"].map(lambda t: B[("last3", t)][0]), lf["team"].map(lambda t: B[("last3", t)][1]))
    lf = lf.assign(out1=(lf["last1"] < lo1) | (lf["last1"] > hi1), out3=(lf["last3"] < lo3) | (lf["last3"] > hi3)).merge(fzs()[["frame_idx", "ok_clip"]], on="frame_idx")
    cl = lf[lf["ok_clip"]].dropna(subset=["last1", "last3"])
    examples = []
    kept = cl[~cl["out1"] & ~cl["out3"]]
    c = kept[kept["team"] == LTR_TEAM]
    if len(c):
        r = c.assign(d=(c["last3"] - B[("last3", LTR_TEAM)][2]).abs()).nsmallest(1, "d").iloc[0]
        examples.append(make_example(r["frame_idx"], f"typical frame: {name(LTR_TEAM)}, last-3 line {r['last3']:.1f} m (median {B[('last3', LTR_TEAM)][2]:.1f} m), kept", team=LTR_TEAM))
    if len(kept):
        r = kept.nlargest(1, "last3").iloc[0]
        examples.append(make_example(r["frame_idx"], f"highest line that is still kept: {name(int(r['team']))}, last-3 {r['last3']:.1f} m", team=int(r["team"])))
    out = cl[cl["out1"] | cl["out3"]]
    if len(out):
        r = out.assign(z=lambda d: np.maximum((d["last3"] - d["team"].map(lambda t: B[("last3", t)][2])).abs(), (d["last1"] - d["team"].map(lambda t: B[("last1", t)][2])).abs())).nlargest(1, "z").iloc[0]
        examples.append(make_example(r["frame_idx"], f"OUTLIER removed from the statistics: {name(int(r['team']))}, last-1 {r['last1']:.1f} m / last-3 {r['last3']:.1f} m", team=int(r["team"])))
    else:
        print("no frame is removed by the 1.5 x IQR rule in this match; showing the frame closest to the limit instead")
        r = kept.assign(z=lambda d: d["last3"] - d["team"].map(lambda t: B[("last3", t)][1])).nlargest(1, "z").iloc[0]
        examples.append(make_example(r["frame_idx"], f"no outlier in this match: frame closest to the IQR limit ({name(int(r['team']))}, last-3 {r['last3']:.1f} m)", team=int(r["team"])))

    def builder(ctx, ex):
        di = defenders_sorted(ctx)
        if di is None:
            return dict(hud=["(this frame is not in the export: the team is not out of possession here)"])
        t, d = di
        l1, l3 = d["dist"].iloc[0], d["dist"].iloc[:3].mean(); col = TEAM_RGB[t]
        b1, b3 = B[("last1", t)], B[("last3", t)]
        o1, o3 = (l1 < b1[0] or l1 > b1[1]), (l3 < b3[0] or l3 > b3[1])
        ops = [("line", (raw_x(t, l1), 0), (raw_x(t, l1), PITCH_WIDTH), col, 0.95, 3), ("text", (raw_x(t, l1), PITCH_WIDTH * 0.92), f"last-1 {l1:.1f} m", col),
               ("line", (raw_x(t, l3), 0), (raw_x(t, l3), PITCH_WIDTH), (255, 255, 255), 0.9, 5), ("text", (raw_x(t, l3), PITCH_WIDTH * 0.08), f"last-3 {l3:.1f} m", (90, 90, 90))]
        hl = {int(r.track_id): (col, f"L{i + 1}") for i, r in enumerate(d.iloc[:3].itertuples())}
        hud = [f"{name(t)} out of possession",
               f"last-1: {l1:.1f} m   kept range [{b1[0]:.1f}, {b1[1]:.1f}]   median {b1[2]:.1f}  ->  " + ("OUTLIER, removed" if o1 else "kept"),
               f"last-3: {l3:.1f} m   kept range [{b3[0]:.1f}, {b3[1]:.1f}]   median {b3[2]:.1f}  ->  " + ("OUTLIER, removed" if o3 else "kept")]
        return dict(ops=ops, hl=hl, hud=hud, chip=("OUTLIER REMOVED", BAD_RGB) if (o1 or o3) else ("KEPT", OK_RGB))

    verify_metric("m09_line_height", "Metric 9 | defensive line height (last-1 / last-3, IQR)", dedupe(examples), builder)

else:
    print("real-frame check skipped:", "; ".join(VERIFY_PROBLEMS))


## 10. Team compactness while out of possession
Per frame, out-of-possession outfield block of the defending team:
- **depth** = mean distance from own goal of the front 3 outfield players minus that of the back 3 (needs at least 6 tracked outfield players)
- **width** = lateral spread (max y - min y) after removing outliers with the 1.5 x IQR rule
- **area** = depth x width (m²). Smaller = more compact block.

In [ ]:
sh = team_shape()
rows = []
for team in TEAMS:
    s = sh[sh["team"] == team]
    row = dict(team=name(team), frames=len(s))
    for col, lab in [("depth_m", "depth"), ("width_m", "width"), ("area_m2", "area")]:
        row.update({f"{lab}_mean": round(s[col].mean(), 1), f"{lab}_median": round(s[col].median(), 1), f"{lab}_std": round(s[col].std(), 1)})
    rows.append(row)
t10 = pd.DataFrame(rows).set_index("team")
display(t10)

A, B = TEAMS[:2]
aa, ab = t10.loc[name(A), "area_mean"], t10.loc[name(B), "area_mean"]
tight, loose = (A, B) if aa <= ab else (B, A)
print("\nSUMMARY")
for team in TEAMS:
    r = t10.loc[name(team)]
    print(f"  {name(team)}: block {r['depth_mean']:.1f} m deep and {r['width_mean']:.1f} m wide on average -> {r['area_mean']:.0f} m² (median {r['area_median']:.0f}).")
print(f"  {name(tight)} defend in the more compact block, about {100 * abs(aa - ab) / max(aa, ab):.0f}% smaller by area than {name(loose)}.")

#### Real frames: metric 10
The out-of-possession block on its tightest, a typical and its loosest frame: shaded rectangle = depth x width.

In [ ]:
# Real-frame check, metric 10: the out-of-possession block (depth x width = area) on its tightest, a typical and its loosest frame.
if VERIFY_READY:
    sh = team_shape().merge(fzs()[["frame_idx", "ok_clip"]], on="frame_idx")
    avg = team_shape().groupby("team")["area_m2"].mean().to_dict()
    sh = sh[sh["ok_clip"]]
    examples = []
    if len(sh):
        r = sh.nsmallest(1, "area_m2").iloc[0]
        examples.append(make_example(r["frame_idx"], f"tightest block of the match: {name(int(r['team']))}, {r['area_m2']:.0f} m2", team=int(r["team"])))
        c = sh[sh["team"] == LTR_TEAM]
        if len(c):
            r = c.assign(d=(c["area_m2"] - avg[LTR_TEAM]).abs()).nsmallest(1, "d").iloc[0]
            examples.append(make_example(r["frame_idx"], f"typical block: {name(LTR_TEAM)}, {r['area_m2']:.0f} m2 (match average {avg[LTR_TEAM]:.0f} m2)", team=LTR_TEAM))
        r = sh.nlargest(1, "area_m2").iloc[0]
        examples.append(make_example(r["frame_idx"], f"loosest block of the match: {name(int(r['team']))}, {r['area_m2']:.0f} m2", team=int(r["team"])))

    def builder(ctx, ex):
        g = block_geometry(ctx)
        if g is None:
            return dict(hud=["(not enough tracked outfield players, or the team is not out of possession here)"])
        t = g["team"]
        hud = [f"{name(t)} out of possession: block = {g['depth']:.1f} m deep x {g['width']:.1f} m wide = {g['area']:.0f} m2",
               f"depth: front 3 at {g['front']:.1f} m minus back 3 at {g['back']:.1f} m from own goal | width: IQR-trimmed spread of y",
               f"{name(t)} match average block: {avg[t]:.0f} m2 (this frame {g['area'] - avg[t]:+.0f} m2)"]
        hl = {int(r.track_id): (TEAM_RGB[t], "") for r in g["players"].iloc[:PRM["BACK_N"]].itertuples()}
        hl.update({int(r.track_id): (TEAM_RGB[t], "") for r in g["players"].iloc[-PRM["FRONT_N"]:].itertuples()})
        return dict(ops=block_ops(g), hl=hl, hud=hud, chip=(f"BLOCK {g['area']:.0f} m2", TEAM_RGB[t]))

    verify_metric("m10_compactness", "Metric 10 | team compactness out of possession", dedupe(examples), builder)

else:
    print("real-frame check skipped:", "; ".join(VERIFY_PROBLEMS))


## 11. Compactness while actively pressing vs not
Same shape metrics as section 10, split by whether the team is pressing the ball carrier in that frame (`under_pressure` = True) or sitting in its resting out-of-possession shape (False). This shows how the block changes when it presses; it does not by itself show that pressing causes the change.

In [ ]:
sh = team_shape().dropna(subset=["under_pressure"]).copy()
sh["state"] = np.where(sh["under_pressure"].astype(bool), "pressing", "not pressing")
rows = []
for team in TEAMS:
    for state in ["pressing", "not pressing"]:
        s = sh[(sh["team"] == team) & (sh["state"] == state)]
        rows.append(dict(team=name(team), state=state, frames=len(s),
                         depth_mean=round(s["depth_m"].mean(), 1), width_mean=round(s["width_m"].mean(), 1),
                         area_mean=round(s["area_m2"].mean(), 1), area_median=round(s["area_m2"].median(), 1), area_std=round(s["area_m2"].std(), 1)))
t11 = pd.DataFrame(rows).set_index(["team", "state"])
display(t11)

print("\nSUMMARY")
for team in TEAMS:
    p, n = t11.loc[(name(team), "pressing")], t11.loc[(name(team), "not pressing")]
    if p["frames"] == 0 or n["frames"] == 0:
        print(f"  {name(team)}: not enough frames in one of the two states."); continue
    d = p["area_mean"] - n["area_mean"]
    word = "tighter" if d < 0 else "looser"
    print(f"  {name(team)}: {p['area_mean']:.0f} m² while pressing vs {n['area_mean']:.0f} m² when not -> the block is {abs(d):.0f} m² ({100 * abs(d) / n['area_mean']:.0f}%) {word} while pressing "
          f"(depth {p['depth_mean']:.1f} vs {n['depth_mean']:.1f} m, width {p['width_mean']:.1f} vs {n['width_mean']:.1f} m).")

#### Real frames: metric 11
The same block while pressing and while not pressing.

In [ ]:
# Real-frame check, metric 11: the same block while the team is pressing and while it is not.
if VERIFY_READY:
    sh = team_shape().dropna(subset=["under_pressure"]).copy()
    sh["state"] = np.where(sh["under_pressure"].astype(bool), "pressing", "not pressing")
    mean_area = sh.groupby(["team", "state"])["area_m2"].mean()
    z = fzs()[["frame_idx", "stable_press", "stable_nopress", "ok_clip"]]
    sh = sh.merge(z, on="frame_idx")
    def closest(team, state):
        flag = "stable_press" if state == "pressing" else "stable_nopress"
        c = sh[(sh["team"] == team) & (sh["state"] == state) & sh[flag] & sh["ok_clip"]]
        return None if c.empty else c.assign(d=(c["area_m2"] - mean_area[(team, state)]).abs()).nsmallest(1, "d").iloc[0]
    delta = {t: abs(mean_area.get((t, "pressing"), np.nan) - mean_area.get((t, "not pressing"), np.nan)) for t in TEAMS}
    focus = max(delta, key=lambda t: -1 if np.isnan(delta[t]) else delta[t])
    other = [t for t in TEAMS if t != focus][0]
    examples = []
    for team, state in [(focus, "pressing"), (focus, "not pressing"), (other, "pressing")]:
        r = closest(team, state)
        if r is not None:
            examples.append(make_example(r["frame_idx"], f"{name(team)} {state}: block {r['area_m2']:.0f} m2 (its average while {state}: {mean_area[(team, state)]:.0f} m2)", team=team))

    def builder(ctx, ex):
        g = block_geometry(ctx)
        if g is None:
            return dict(hud=["(not enough tracked outfield players, or the team is not out of possession here)"])
        t = g["team"]; ops, hl = press_ops(ctx)
        hud = [f"{name(t)}: block {g['depth']:.1f} m x {g['width']:.1f} m = {g['area']:.0f} m2",
               f"average while pressing {mean_area.get((t, 'pressing'), np.nan):.0f} m2  |  while not pressing {mean_area.get((t, 'not pressing'), np.nan):.0f} m2"]
        return dict(ops=block_ops(g) + ops, hl=hl, hud=hud, chip=("PRESSING", PRESS_RGB) if ctx.under else ("NOT PRESSING", (90, 130, 90)))

    verify_metric("m11_compactness_pressing", "Metric 11 | compactness while pressing vs not", dedupe(examples), builder)

else:
    print("real-frame check skipped:", "; ".join(VERIFY_PROBLEMS))


## 12. Third x channel grid
Where each team's pressure happens: the thirds that count as pressing (middle and attacking, in the pressing team's attacking direction) x three lateral channels (thirds of the pitch width), 6 cells. Each cell is the % of the team's pressure frames. `left` / `right` follow `LEFT_IS_LOW_Y_ATT` from your build-up notebook (looking towards the opponent's goal); flip that switch in the setup cell if they look mirrored on the video.

In [ ]:
fz = frames_zoned()
pr_ = fz[fz["under_pressure"]].dropna(subset=["zone", "channel"])

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
rows = []
for ax, team in zip(axes, FIG_TEAMS):
    s = pr_[pr_["def_team"] == team]
    cnt = s.groupby(["channel", "zone"]).size().unstack(fill_value=0).reindex(index=CH_ORDER, columns=PRESS_ZONES, fill_value=0)
    tot = int(cnt.values.sum())
    pct = 100 * cnt / tot if tot else cnt * np.nan
    draw_grid(ax, pct.to_numpy(), [[f"{pct.iloc[i, j]:.1f}%\n{cnt.iloc[i, j]} fr" for j in range(len(PRESS_ZONES))] for i in range(len(CH_ORDER))],
              f"{name(team)}: % of pressure frames ({tot} total)", cmap=TEAM_CMAP[team], rtl=(team != LTR_TEAM))
    for ch in CH_ORDER:
        for z in PRESS_ZONES:
            rows.append(dict(team=name(team), third=z, channel=CH_LABEL[ch], frames=int(cnt.loc[ch, z]), seconds=round(cnt.loc[ch, z] / FPS, 1),
                             pct_of_team_pressure=round(pct.loc[ch, z], 1)))
plt.tight_layout(); plt.show()
t12 = pd.DataFrame(rows)
display(t12.pivot_table(index=["team", "channel"], columns="third", values="pct_of_team_pressure", sort=False)[PRESS_ZONES])

print("SUMMARY")
for team in TEAMS:
    s = t12[t12["team"] == name(team)]
    top = s.loc[s["pct_of_team_pressure"].idxmax()]
    by_third = s.groupby("third")["pct_of_team_pressure"].sum().reindex(PRESS_ZONES)
    by_ch = s.groupby("channel")["pct_of_team_pressure"].sum()
    print(f"  {name(team)}: biggest cell = {top['channel']} channel of the {top['third'].replace('_', ' ')} ({top['pct_of_team_pressure']:.1f}% of their pressure). "
          f"By third ({'/'.join(z.replace('_third', '') for z in PRESS_ZONES)}): {'/'.join(f'{v:.0f}' for v in by_third)}%. "
          f"By channel left/centre/right: {by_ch['left']:.0f}/{by_ch['centre']:.0f}/{by_ch['right']:.0f}%.")

#### Real frames: metric 12
Pressing in the biggest third x channel cell of each team and in the biggest attacking-third cell (shaded).

In [ ]:
# Real-frame check, metric 12: the third x channel grid. The biggest cell of each team and the biggest attacking-third cell are shaded.
if VERIFY_READY:
    fz = frames_zoned(); pz = fz[fz["under_pressure"]].dropna(subset=["zone", "channel"])
    cells = {}
    for team in TEAMS:
        s = pz[pz["def_team"] == team]; tot = len(s)
        for (zone, ch), n in s.groupby(["zone", "channel"]).size().items():
            cells[(team, zone, ch)] = (n, 100 * n / tot)
    def cell_example(team, zone, ch, tag):
        n, pct = cells[(team, zone, ch)]
        f = pick_frame(True, team=team, extra=lambda z: (z["zone"] == zone) & (z["channel"] == ch))
        if f is None:
            return None
        return make_example(f, f"{name(team)}, {tag}: {zone.replace('_', ' ')} / {CH_LABEL[ch]} channel ({pct:.1f}% of its pressure)", team=team, rect=cell_rect(team, zone, ch),
                            hud=[f"cell {zone.replace('_', ' ')} / {CH_LABEL[ch]} (as seen by {name(team)}): {int(n)} pressure frames = {pct:.1f}% of {name(team)}'s pressure"])
    def top(team, zone=None):
        c = [(v[0], k) for k, v in cells.items() if k[0] == team and (zone is None or k[1] == zone)]
        return max(c)[1] if c else None
    examples = []
    for team in FIG_TEAMS:
        k = top(team)
        if k:
            examples.append(cell_example(*k, "biggest cell"))
    more = max(TEAMS, key=lambda t: sum(v[0] for k, v in cells.items() if k[0] == t))
    k = top(more, "attacking_third")
    if k:
        examples.append(cell_example(*k, "biggest attacking-third cell"))
    verify_metric("m12_third_channel_grid", "Metric 12 | third x channel grid", dedupe(examples), b_press)

else:
    print("real-frame check skipped:", "; ".join(VERIFY_PROBLEMS))


## 13. Zone intensity per time in zone
Sections 3 and 12 count pressure frames, but a team that spends more time defending in a zone naturally has more pressure there. This normalises by the time the ball is in the zone while the team is out of possession (only the middle and attacking third, since pressure in the team's own third is not counted):
`zone intensity % = pressing frames with the ball in the zone / out-of-possession frames with the ball in the zone`. Cells with fewer than `MIN_CELL_FRAMES` ball-in-cell frames are greyed out.

In [ ]:
fz = frames_zoned().dropna(subset=["zone", "channel"])
minf = PRM["MIN_CELL_FRAMES"]

# ---- by third
g = fz.groupby(["def_team", "zone"]).agg(ball_frames=("under_pressure", "size"), pressing_frames=("under_pressure", "sum")).reset_index()
rows = []
for team in TEAMS:
    tot_ball = int(g.loc[g["def_team"] == team, "ball_frames"].sum())
    for z in PRESS_ZONES:
        r = g[(g["def_team"] == team) & (g["zone"] == z)]
        nb_, npf = (int(r["ball_frames"].iloc[0]), int(r["pressing_frames"].iloc[0])) if len(r) else (0, 0)
        rows.append(dict(team=name(team), third=z, ball_in_zone_sec=round(nb_ / FPS, 1), pct_of_out_of_possession_time=round(100 * nb_ / tot_ball, 1) if tot_ball else np.nan,
                         pressing_sec=round(npf / FPS, 1), zone_intensity_pct=round(100 * npf / nb_, 1) if nb_ >= minf else np.nan))
t13 = pd.DataFrame(rows).set_index(["team", "third"])
display(t13)

# ---- by third x channel
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for ax, team in zip(axes, FIG_TEAMS):
    s = fz[fz["def_team"] == team]
    n = s.groupby(["channel", "zone"]).size().unstack(fill_value=0).reindex(index=CH_ORDER, columns=PRESS_ZONES, fill_value=0)
    p = s[s["under_pressure"]].groupby(["channel", "zone"]).size().unstack(fill_value=0).reindex(index=CH_ORDER, columns=PRESS_ZONES, fill_value=0)
    inten = (100 * p / n.replace(0, np.nan)).where(n >= minf)
    draw_grid(ax, inten.to_numpy(), [["n/a" if pd.isna(inten.iloc[i, j]) else f"{inten.iloc[i, j]:.0f}%\n({n.iloc[i, j] / FPS:.0f} s)" for j in range(len(PRESS_ZONES))] for i in range(len(CH_ORDER))],
              f"{name(team)}: % of ball-in-cell time spent pressing", cmap=TEAM_CMAP[team], vmax=100, rtl=(team != LTR_TEAM))
plt.tight_layout(); plt.show()

print("SUMMARY")
for team in TEAMS:
    r = t13.loc[name(team)].dropna(subset=["zone_intensity_pct"])
    if r.empty:
        print(f"  {name(team)}: not enough frames."); continue
    hi, lo = r["zone_intensity_pct"].idxmax(), r["zone_intensity_pct"].idxmin()
    print(f"  {name(team)}: presses most when the ball is in the {hi.replace('_', ' ')} ({r.loc[hi, 'zone_intensity_pct']:.0f}% of that time) and least in the {lo.replace('_', ' ')} "
          f"({r.loc[lo, 'zone_intensity_pct']:.0f}%). Time with the ball in the {'/'.join(z.replace('_third', '') for z in PRESS_ZONES)} third: "
          + "/".join(f"{t13.loc[(name(team), z), 'pct_of_out_of_possession_time']:.0f}%" for z in PRESS_ZONES) + " of their out-of-possession time.")

#### Real frames: metric 13
The highest-intensity cell with a pressing frame and with a frame without pressing, and the lowest-intensity cell.

In [ ]:
# Real-frame check, metric 13: zone intensity per time in zone. Frames inside the highest- and lowest-intensity cells, pressing and not pressing.
if VERIFY_READY:
    fz = frames_zoned().dropna(subset=["zone", "channel"]); minf = PRM["MIN_CELL_FRAMES"]
    g = fz.groupby(["def_team", "zone", "channel"]).agg(n=("under_pressure", "size"), p=("under_pressure", "sum")).reset_index()
    g = g[g["zone"].isin(PRESS_ZONES) & (g["n"] >= minf)].assign(inten=lambda d: 100 * d["p"] / d["n"])
    def cell_example(r, pressing, tag):
        team, zone, ch = int(r["def_team"]), r["zone"], r["channel"]
        f = pick_frame(pressing, team=team, extra=lambda z: (z["zone"] == zone) & (z["channel"] == ch))
        if f is None:
            return None
        return make_example(f, f"{tag}: {name(team)}, {zone.replace('_', ' ')} / {CH_LABEL[ch]} cell, intensity {r['inten']:.0f}% ({'pressing' if pressing else 'not pressing'} frame)", team=team,
                            rect=cell_rect(team, zone, ch), hud=[f"cell intensity {r['inten']:.0f}% = pressing in {int(r['p'])} of {int(r['n'])} frames ({r['n'] / FPS:.0f} s) with the ball in the cell"])
    examples = []
    if len(g):
        hi, lo = g.loc[g["inten"].idxmax()], g.loc[g["inten"].idxmin()]
        examples = [cell_example(hi, True, "highest-intensity cell"), cell_example(hi, False, "same cell, a frame WITHOUT pressing"), cell_example(lo, False, "lowest-intensity cell")]
    verify_metric("m13_zone_intensity", "Metric 13 | zone intensity per time in zone", dedupe(examples, 5), b_press)

else:
    print("real-frame check skipped:", "; ".join(VERIFY_PROBLEMS))


## 14. Press-to-regain rate
A **pressing episode** (own-third pressure is not counted, so every episode starts in the middle or attacking third) = a run of pressing frames (`under_pressure`) inside one possession spell, gaps of up to `EPISODE_GAP_FRAMES` bridged, lasting at least `MIN_EPISODE_SEC`. It counts as a **regain** if the pressing team becomes the ball carrier within `REGAIN_WINDOW_SEC` (5 s) after the episode ends. `regained and held` is stricter: the pressing team also starts a stable possession spell (holds the ball 1.5 s or more) in that window.

In [ ]:
W, GAP = F(PRM["REGAIN_WINDOW_SEC"]), PRM["EPISODE_GAP_FRAMES"]
eps = press_episodes().copy()
assert len(eps), "no pressing episodes found"

def regain_table(df, by):
    rows = []
    for team in TEAMS:
        for key, s in df[df["team"] == team].groupby(by, observed=True):
            rows.append({"team": name(team), by: key, "episodes": len(s), "regained": int(s["regained"].sum()), "regain_pct": round(100 * s["regained"].mean(), 1),
                         "regained_and_held": int(s["regained_and_held"].sum()), "regain_and_held_pct": round(100 * s["regained_and_held"].mean(), 1)})
    return pd.DataFrame(rows).set_index(["team", by])

eps["all"] = "all episodes"
eps["pressers"] = np.where(eps["max_pressers"] >= 2, "2+ pressers", "1 presser")
eps["start_third"] = pd.Categorical(eps["start_zone"], categories=PRESS_ZONES, ordered=True)
print(f"{len(eps)} pressing episodes (>= {PRM['MIN_EPISODE_SEC']} s, gaps <= {GAP} frames bridged); regain window {PRM['REGAIN_WINDOW_SEC']:.0f} s\n")
print("overall:"); t14 = regain_table(eps, "all"); display(t14)
print("by number of pressers (max at the same time in the episode):"); display(regain_table(eps, "pressers"))
print("by third where the episode started (pressing team's attacking frame):"); display(regain_table(eps, "start_third"))
print("episode length (s):"); display(eps.assign(sec=eps["n_frames"] / FPS).groupby(eps["team"].map(name))["sec"].describe().round(1)[["count", "mean", "50%", "max"]])

print("SUMMARY")
for team in TEAMS:
    r = t14.loc[(name(team), "all episodes")]
    print(f"  {name(team)}: {int(r['episodes'])} pressing episodes, ball back within {PRM['REGAIN_WINDOW_SEC']:.0f} s after {int(r['regained'])} ({r['regain_pct']:.1f}%); "
          f"{int(r['regained_and_held'])} ({r['regain_and_held_pct']:.1f}%) were regained and held.")
print("  There is no baseline here: a regain rate says how often a press is followed by the ball, not how much of that the press caused.")
if (t14["episodes"] < 15).any():
    print("  CAUTION: fewer than 15 episodes for a team, the percentages are unstable.")

#### Real frames: metric 14
Two pressing episodes that ended with the ball regained and one that did not; the 5 s regain window counts down on screen.

In [ ]:
# Real-frame check, metric 14: press-to-regain. Two pressing episodes that ended with the ball regained and one that did not; the regain window is shown on the frame.
if VERIFY_READY:
    eps = press_episodes(); RW = PRM["REGAIN_WINDOW_SEC"]
    team_of = roster.drop_duplicates("track_id").set_index("track_id")["team"].to_dict()
    stats = {t: (int(len(eps[eps["team"] == t])), int(eps.loc[eps["team"] == t, "regained"].sum())) for t in TEAMS}
    def mid(df):
        return None if df.empty else df.sort_values("end_frame").iloc[len(df) // 2]
    ok_ep = eps[eps["n_frames"] >= F(1.0)]
    reg, notreg = ok_ep[ok_ep["regained"]], ok_ep[~ok_ep["regained"]]
    more = max(TEAMS, key=lambda t: len(reg[reg["team"] == t]))
    picks = [("regained", mid(reg[reg["team"] == more])), ("regained", mid(reg[reg["team"] != more])), ("NOT regained", mid(notreg))]
    examples = []
    for label, r in picks:
        if r is None:
            continue
        T = int(r["team"]); end = int(r["end_frame"]); rf = None if pd.isna(r["regain_frame"]) else int(r["regain_frame"])
        f1 = (rf + F(1.0)) if rf is not None else end + F(RW) + F(0.5)
        examples.append(make_example(end, f"{name(T)} press episode ({r['n_frames'] / FPS:.1f} s), ball {label}", f0=max(int(r["start_frame"]) - F(0.5), end - F(4)), f1=f1, max_sec=10,
                                     team=T, start=int(r["start_frame"]), end=end, regain=rf))

    def builder(ctx, ex):
        f, T, end, rf = ctx.f, ex["team"], ex["end"], ex["regain"]
        t_after = (f - end) / FPS; ops, hl = ([], {})
        if f <= end:
            ops, hl = press_ops(ctx); chip = (f"PRESSING {name(T)}", PRESS_RGB)
        elif rf is not None and f >= rf:
            chip = (f"BALL REGAINED +{(rf - end) / FPS:.1f} s", OK_RGB)
            if ctx.carrier is not None and team_of.get(ctx.carrier) == T:
                hl[ctx.carrier] = (OK_RGB, "regains")
        elif t_after <= RW:
            chip = (f"REGAIN WINDOW {max(RW - t_after, 0):.1f} s", (230, 160, 30))
        else:
            chip = ("NOT REGAINED", BAD_RGB)
        n, k = stats[T]
        hud = [f"{name(T)} pressing episode: {mmss_clock(ex['start'])} to {mmss_clock(end)} ({(end - ex['start']) / FPS:.1f} s)",
               f"regain = {name(T)} becomes the ball carrier within {RW:.0f} s after the episode ends: " + (f"YES, after {(rf - end) / FPS:.1f} s" if rf is not None else "NO"),
               f"{name(T)} overall: {k} of {n} episodes regained ({100 * k / max(n, 1):.1f}%)"]
        return dict(ops=ops, hl=hl, hud=hud, chip=chip)

    verify_metric("m14_press_to_regain", "Metric 14 | press-to-regain", dedupe(examples, 3), builder)

else:
    print("real-frame check skipped:", "; ".join(VERIFY_PROBLEMS))


## 15. Pressing time per player and share of the team's pressing
A defender counts as pressing in a frame when his own press flag is on (within 5 m of the ball carrier and within 2 m or closing at 1.5 m/s or more) **and** the team is in a sustained pressing phase (`under_pressure`), so one-frame flickers are ignored. Per player:
- `share_of_team_presser_time_pct`: his pressing frames / all pressing frames of all his team's players (sums to 100%; several players press at once)
- `involved_in_pct_of_team_pressing`: the % of the team's pressing frames in which he was one of the pressers
- `pressing_rate_pct`: his pressing frames / the frames he was tracked while his team defended (fair to players with less time on the pitch)

In [ ]:
d = player_press_frames()
t15 = player_press_table()
team_press_frames = frames[frames["under_pressure"]].groupby("def_team").size()
for team in TEAMS:
    print(f"\n{name(team)}  (team pressing time {team_press_frames.get(team, 0) / FPS:.0f} s; players listed by pressing time)")
    display(t15.loc[team].sort_values("pressing_sec", ascending=False))

print("\nSUMMARY")
for team in TEAMS:
    s = t15.loc[team].sort_values("pressing_sec", ascending=False)
    top = s.head(3)
    print(f"  {name(team)}: most pressing time = " + ", ".join(f"{p} ({r['pressing_sec']:.0f} s, {r['share_of_team_presser_time_pct']:.0f}% of the team's presser time)" for p, r in top.iterrows())
          + f". The top three account for {top['share_of_team_presser_time_pct'].sum():.0f}% of it.")
    hi = s[s["defending_sec"] >= 300].sort_values("pressing_rate_pct", ascending=False).head(1)
    if len(hi):
        print(f"      highest pressing rate (at least 5 min tracked while defending): {hi.index[0]} at {hi['pressing_rate_pct'].iloc[0]:.1f}% of his defending time.")

#### Real frames: metric 15
The player with the most pressing time of each team, pressing on real frames, with his share of the team's pressing.

In [ ]:
# Real-frame check, metric 15: pressing time per player. The player with the most pressing time of each team, pressing on real frames.
if VERIFY_READY:
    t15 = player_press_table().reset_index()
    team_tot = t15.groupby("team")["pressing_sec"].sum()
    def top(team, rank=0):
        c = t15[t15["team"] == team].sort_values("pressing_sec", ascending=False)
        return c.iloc[rank] if len(c) > rank else None
    more = int(team_tot.idxmax())
    examples = []
    for team, rank in [(LTR_TEAM, 0), (RTL_TEAM, 0), (more, 1)]:
        r = top(team, rank)
        if r is None:
            continue
        f = player_frame_pick(int(r["team"]), r["player"])
        if f is not None:
            examples.append(make_example(f, f"{name(team)} #{int(r['number'])} {r['player']}: {'most' if rank == 0 else 'second most'} pressing time of his team ({r['pressing_sec']:.1f} s)",
                                         team=int(r["team"]), player=r["player"], num=int(r["number"]), sec=float(r["pressing_sec"]), share=float(r["share_of_team_presser_time_pct"]),
                                         inv=float(r["involved_in_pct_of_team_pressing"]), tid=int(player_press_frames().query("player == @r.player and team == @r.team")["track_id"].iloc[0])))

    def builder(ctx, ex):
        ops, hl = press_ops(ctx, show_lines=False)
        tid, T = ex["tid"], ex["team"]
        pressing = tid in ctx.pressers
        if tid in ctx.pos_xy and ctx.tgt is not None and pressing:
            p = ctx.pos_xy[tid]; ops.append(("line", p, ctx.tgt, PRESS_RGB, 0.95, 5))
            hl[tid] = (PRESS_RGB, f"#{ex['num']} {ex['player']}  {np.hypot(p[0] - ctx.tgt[0], p[1] - ctx.tgt[1]):.1f} m")
        else:
            hl[tid] = (GREY_RGB, f"#{ex['num']} {ex['player']}")
        hud = [f"#{ex['num']} {ex['player']} ({name(T)}): pressing time {ex['sec']:.1f} s = {ex['share']:.1f}% of {name(T)}'s presser time",
               f"in the frames where {name(T)} presses, he is one of the pressers in {ex['inv']:.1f}% of them",
               "this frame: " + ("he is pressing the carrier (own press flag inside a sustained pressing phase)" if pressing else "he is not pressing")]
        return dict(ops=ops, hl=hl, hud=hud, chip=("PRESSING" if pressing else "NOT PRESSING", PRESS_RGB if pressing else (90, 130, 90)))

    verify_metric("m15_pressing_time_per_player", "Metric 15 | pressing time per player", dedupe(examples), builder)

else:
    print("real-frame check skipped:", "; ".join(VERIFY_PROBLEMS))


## 16. Where each player presses (third x channel)
For each player, the % of his pressing frames (as defined in section 15) by the third (middle / attacking) and channel where the ball carrier was, in his team's attacking frame. Players with fewer than `MIN_PLAYER_PRESS_FRAMES` pressing frames are greyed out in the figure: too little to read.

In [ ]:
cells = [(z, ch) for z in PRESS_ZONES for ch in CH_ORDER]
col_name = lambda z, ch: f"{ZONE_SHORT[z]}_{CH_LABEL[ch]}"
t16 = player_cell_table()
display(t16.drop(columns=["main_zone", "main_channel"]))

for team in FIG_TEAMS:
    cmap = TEAM_CMAP[team]
    ts = t16.loc[name(team)]
    n_pl = len(ts); ncols = 5; nrows = int(np.ceil(n_pl / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.1 * ncols, 2.7 * nrows), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, (player, r) in zip(axes.ravel(), ts.iterrows()):
        ax.axis("on")
        M = np.array([[r[col_name(z, ch)] for z in PRESS_ZONES] for ch in CH_ORDER], dtype=float)
        if not r["enough_data"]:
            M = np.full((len(CH_ORDER), len(PRESS_ZONES)), np.nan)
        draw_grid(ax, M, [["" if np.isnan(M[i, j]) else f"{M[i, j]:.0f}" for j in range(len(PRESS_ZONES))] for i in range(len(CH_ORDER))],
                  f"{player} ({r['pressing_sec']:.0f} s)" + ("" if r["enough_data"] else "  few frames"), cmap=cmap, vmax=60, rtl=(team != LTR_TEAM))
    fig.suptitle(f"{name(team)}: where each player presses (% of his pressing frames; third x channel)", fontsize=12)
    plt.tight_layout(); plt.show()

print("SUMMARY")
for team in TEAMS:
    ts = t16.loc[name(team)]; ok = ts[ts["enough_data"]]
    if ok.empty:
        print(f"  {name(team)}: no player with enough pressing frames."); continue
    hi = ok.sort_values("pct_attacking_third", ascending=False).head(3)
    lo = ok.sort_values("pct_attacking_third").head(2)
    print(f"  {name(team)}: most of their pressing in the attacking third = " + ", ".join(f"{p} ({r['pct_attacking_third']:.0f}%)" for p, r in hi.iterrows())
          + "; least = " + ", ".join(f"{p} ({r['pct_attacking_third']:.0f}%)" for p, r in lo.iterrows()) + ".")
    print("      most common cell per player: " + "; ".join(f"{p}: {r['main_cell']}" for p, r in ok.sort_values("pressing_sec", ascending=False).head(5).iterrows()) + " (top 5 by pressing time).")

#### Real frames: metric 16
Players pressing in their main cell (shaded): the highest attacking-third share of each team and a top presser's main cell.

In [ ]:
# Real-frame check, metric 16: where each player presses. Players with the highest share of attacking-third pressing, and the main cell of a top presser.
if VERIFY_READY:
    t16 = player_cell_table().reset_index(); ok = t16[t16["enough_data"]]
    d_all = player_press_frames(); inv_ch = {v: k for k, v in CH_LABEL.items()}
    tid_of = lambda team, player: int(d_all[(d_all["team"] == team) & (d_all["player"] == player)]["track_id"].iloc[0])
    def team_id(nm):
        return next(t for t in TEAMS if name(t) == nm)
    def example(r, zone, ch, tag):
        team = team_id(r["team"]); f = player_frame_pick(team, r["player"], zone=zone, channel=ch)
        if f is None:
            return None
        share = float(r[f"{ZONE_SHORT[zone]}_{CH_LABEL[ch]}"])
        return make_example(f, f"{r['team']} {r['player']}: {tag}, {zone.replace('_', ' ')} / {CH_LABEL[ch]} cell ({share:.0f}% of his pressing)", team=team, player=r["player"], tid=tid_of(team, r["player"]),
                            sec=float(r["pressing_sec"]), share=share, rect=cell_rect(team, zone, ch), cell=f"{zone.replace('_', ' ')} / {CH_LABEL[ch]}")
    examples = []
    for team in FIG_TEAMS:
        c = ok[ok["team"] == name(team)]
        if len(c):
            r = c.sort_values("pct_attacking_third", ascending=False).iloc[0]
            att_cols = {chn: f"att_{CH_LABEL[chn]}" for chn in CH_ORDER}
            best = max(CH_ORDER, key=lambda chn: r[att_cols[chn]])
            examples.append(example(r, "attacking_third", best, f"most attacking-third pressing of his team ({r['pct_attacking_third']:.0f}%)"))
    top_team = max(TEAMS, key=lambda t: ok.loc[ok["team"] == name(t), "pressing_sec"].sum())
    c = ok[ok["team"] == name(top_team)].sort_values("pressing_sec", ascending=False)
    if len(c):
        r = c.iloc[0]; examples.append(example(r, r["main_zone"], r["main_channel"], "top presser's main cell"))

    def builder(ctx, ex):
        ops, hl = press_ops(ctx, show_lines=False)
        tid, T = ex["tid"], ex["team"]; pressing = tid in ctx.pressers
        if ctx.fr is not None and ex.get("rect") is not None:
            r = ex["rect"]; ops.insert(0, ("rect", r, TEAM_RGB[T], 0.12, 0.95))
        if tid in ctx.pos_xy and ctx.tgt is not None and pressing:
            p = ctx.pos_xy[tid]; ops.append(("line", p, ctx.tgt, PRESS_RGB, 0.95, 5)); hl[tid] = (PRESS_RGB, ex["player"])
        else:
            hl[tid] = (GREY_RGB, ex["player"])
        inside = ctx.tgt is not None and ex["rect"][0] <= ctx.tgt[0] <= ex["rect"][2] and ex["rect"][1] <= ctx.tgt[1] <= ex["rect"][3]
        hud = [f"{ex['player']} ({name(T)}): {ex['share']:.0f}% of his {ex['sec']:.0f} s of pressing happens in the shaded cell ({ex['cell']}, as seen by {name(T)})",
               "this frame: " + ("pressing" if pressing else "not pressing") + f", ball {'INSIDE' if inside else 'outside'} the shaded cell"]
        return dict(ops=ops, hl=hl, hud=hud, chip=("PRESSING" if pressing else "NOT PRESSING", PRESS_RGB if pressing else (90, 130, 90)))

    verify_metric("m16_player_press_cells", "Metric 16 | where each player presses", dedupe(examples), builder)

else:
    print("real-frame check skipped:", "; ".join(VERIFY_PROBLEMS))
